# Forecasting Kebutuhan Uang Tunai ATM: End-to-End

**Kasus bisnis:** setiap hari bank harus memutuskan **berapa uang yang diisi ke setiap ATM**.
- Isi terlalu sedikit → ATM **kehabisan uang** (*cash-out*): nasabah kecewa, kehilangan fee, risiko reputasi.
- Isi terlalu banyak → uang **menganggur** di mesin, padahal menanggung biaya dana (*cost of fund*), biaya asuransi, dan risiko keamanan.

Model meramal **penarikan harian per ATM untuk 14 hari ke depan** (satu siklus perencanaan pengisian), lengkap dengan **interval prediksi**, lalu menerjemahkannya menjadi **rekomendasi jumlah pengisian**.

| Tahap | Isi |
|---|---|
| Data & EDA | 20 ATM, 6 tipe lokasi, harian Jan 2022 – Des 2025; pola mingguan, gajian, Lebaran, libur, tren |
| Uji statistik time series | ADF & KPSS (stasioneritas), uji tren, Kruskal-Wallis/Mann-Whitney untuk efek kalender, ACF/PACF |
| Feature engineering | lag & rolling (log), **binning** tanggal → jendela gajian & fase Lebaran, **one-hot** hari/bulan/bin/tipe lokasi |
| 2 model + patokan | **Holt-Winters (ETS)** per ATM vs **Gradient Boosting global** (rekursif); seasonal naive sebagai patokan |
| Cross-validation | **rolling-origin**: tuning di 2023, pemilihan model di 2024 (26 origin) |
| Backtesting | **2025 dikunci** (26 origin × 14 hari × 20 ATM), termasuk Lebaran 2025 |
| Evaluasi statistik | WAPE, bias, MASE, **cluster bootstrap CI**, **Diebold-Mariano (Newey-West + HLN)**, Wilcoxon, Ljung-Box, cakupan interval |
| Bisnis | simulasi pengisian kas: tingkat cash-out vs biaya uang menganggur → kuantil pengisian optimal |
| Produksi | model JSON → FastAPI `/forecast` → Docker |

> Setiap cell kode didahului kotak **🔄 Alur data** (Input → Proses → Output → Berikutnya).

## 0. Peta Alur Data

```
 data/atm_withdrawals.csv (20 ATM × 1.461 hari)  +  data/atm_master.csv (tipe lokasi, kapasitas)
          │  pivot → wide: index tanggal × kolom ATM, nilai log1p(penarikan)
          ▼
 W (1.461 × 20) ──────────► EDA (3) & uji statistik time series (4): hanya dibaca
          │
          ▼  desain validasi WAKTU (Bagian 5) — tidak ada split acak
 ┌───────────────┬────────────────────────┬───────────────────────────────┐
 │ 2022          │ 2023: TUNING           │ 2024: PEMILIHAN MODEL          │ 2025: UJI AKHIR (dikunci)
 │ riwayat awal  │ 8 origin rolling       │ 26 origin rolling (14 hari)    │ 26 origin rolling (14 hari)
 └───────────────┴────────────────────────┴───────────────────────────────┘
          │  setiap origin: latih dengan data ≤ origin → ramal 14 hari → bandingkan dengan aktual
          ▼
 Fitur (Bagian 6): lag 1–28 & rolling (log) + kalender (binning gajian & Lebaran → one-hot) → 51 kolom
          │
 Model (7): seasonal naive | Holt-Winters per ATM | Gradient Boosting global (rekursif)
          │
 Model statistik (7.1): regresi kalender per tipe lokasi (SE Driscoll-Kraay) + SARIMAX → efek %, p-value, GVIF
          │
 Tuning 2023 (8) → seleksi 2024 (9) → interval dari residual 2024 (10) → uji akhir 2025 (11)
          │
 Diagnostik residual (12) → simulasi pengisian kas (13) → interpretasi (14) → ringkasan (15)
          │
          ▼
 Refit 2022–2025 → model.json + ramalan Jan 2026 (16) → validasi JSON (17) → API /forecast (18)
```

### Siklus hidup variabel utama

| Variabel | Bentuk | Dibuat | Dipakai | Isi |
|---|---|---|---|---|
| `tx`, `master` | 29.220 × 3, 20 × 4 | Bagian 2 | 2–3 | Data mentah & profil ATM |
| `W` | 1.461 × 20 | Bagian 2 | semua | log1p(penarikan), wide |
| `TUNE_ORIGINS`, `SELECT_ORIGINS`, `TEST_ORIGINS` | 8 / 26 / 26 tanggal | Bagian 5 | 8, 9, 11 | Titik mulai ramalan |
| `tuning` | tabel | Bagian 8 | 8 | WAPE tiap konfigurasi di 2023 |
| `bt_2024` | ±21.800 baris | Bagian 9 | 9–10 | Ramalan vs aktual 2024, 3 metode |
| `INTERVALS` | 14 × 5 | Bagian 10 | 11–17, API | Kuantil residual log per horizon |
| `bt_2025` | ±21.800 baris | Bagian 11 | 11–15 | Ramalan vs aktual 2025 (uji akhir) |
| `sim` | tabel | Bagian 13 | 13, 16 | Biaya & cash-out per kebijakan pengisian |
| `final_model` | forecaster | Bagian 16 | 16–17 | Model produksi (data s.d. 31 Des 2025) |
| `model.json` | file | Bagian 16 | 17, API | Model + riwayat 35 hari + interval + metadata |

**Aturan yang dijaga:**
1. Tidak pernah ada informasi masa depan: fitur hari t hanya memakai data ≤ t−1, dan model di setiap origin hanya melihat data ≤ origin.
2. Tahun 2025 tidak dipakai untuk tuning, memilih model, maupun membuat interval. Tahun ini **hanya** untuk uji akhir.
3. Konfigurasi dipilih dengan aturan yang ditulis sebelum melihat hasil: WAPE terendah.

## 1. Import & Konfigurasi

> **🔄 Alur data: Import & Konfigurasi**
>
> - **Input:** Belum ada data.
> - **Proses:**
>   1. Import library: pandas/numpy, matplotlib/seaborn, scipy & statsmodels (ADF, KPSS, STL, ACF, OLS-HAC), scikit-learn (Gradient Boosting).
>   2. Import modul proyek: `calendar_id` (libur & Lebaran), `features` (lag + kalender), `models` (naive, ETS, GBR), `backtest` (rolling-origin), `evaluation` (WAPE, MASE, DM, bootstrap, Ljung-Box), `json_model`.
>   3. Konstanta: horizon 14 hari, jarak antar origin 14 hari, asumsi biaya simpan uang 8%/tahun dan penalti cash-out Rp 2 jt per hari.
> - **Output:** Fungsi & konstanta siap.
> - **Berikutnya:** Baca data.

In [ ]:
import json
import platform
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
import statsmodels.api as sm
from scipy import stats
from sklearn.ensemble import GradientBoostingRegressor
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import STL
from statsmodels.tools.sm_exceptions import InterpolationWarning
from statsmodels.tsa.stattools import adfuller, kpss

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))

from src.backtest import naive_scale, rolling_origin  # noqa: E402
from src.calendar_id import CALENDAR_VALID_UNTIL, DOM_BINS, LEBARAN, LEBARAN_BINS, calendar_frame  # noqa: E402
from src.evaluation import (  # noqa: E402
    cluster_bootstrap_ci,
    coverage_test,
    diebold_mariano,
    forecast_metrics,
    gvif,
    holm_correction,
    interval_quantiles,
    ljung_box,
    metrics_by,
    reference_coding,
    resolve_perfect_collinearity,
    segment_bias,
)
from src.features import FEATURE_COLUMNS, LAG_COLS, build_features, to_wide, training_frame  # noqa: E402
from src.json_model import JsonForecaster, export_model, save_json  # noqa: E402
from src.models import ETSForecaster, GBRForecaster, SeasonalNaive  # noqa: E402

RANDOM_STATE = 42
HORIZON = 14                       # siklus perencanaan pengisian kas
STEP = 14                          # jarak antar origin → jendela ramalan tidak tumpang tindih
CARRY_COST = 0.08                  # biaya menyimpan uang di ATM: dana 6% + asuransi & keamanan 2% per tahun (asumsi)
CASHOUT_PENALTY = 2.0              # kerugian per hari ATM kosong, juta Rp (asumsi: fee transaksi hilang + reputasi)
MODEL_LABELS = {"seasonal_naive": "Seasonal naive", "ets": "Holt-Winters (ETS)", "gbr": "Gradient Boosting"}

DATA_DIR = PROJECT_DIR / "data"
MODEL_DIR = PROJECT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 250)
print(f"Python {platform.python_version()} | scikit-learn {sklearn.__version__} | pandas {pd.__version__}")

## 2. Load Data

> **🔄 Alur data: Load Data & Validasi**
>
> - **Input:** `data/atm_withdrawals.csv`, `data/atm_master.csv`.
> - **Proses:**
>   1. Cek: tidak ada duplikat (tanggal, ATM), penarikan > 0, setiap ATM punya jumlah hari yang sama.
>   2. `to_wide`: pivot → index tanggal harian lengkap × kolom ATM, nilai **log1p(penarikan)**.
>   3. `ATM_TYPES` (ATM → tipe lokasi) dan `CAPACITY` (kapasitas kaset).
> - **Output:** `tx` 29.220 baris (20 ATM × 1.461 hari), `master` 20 ATM: mall 4, kantor 4, perumahan 5, transport hub 3, kampus 2, pasar 2; kapasitas kaset Rp 200–750 jt. `W` **1.461 × 20**, lengkap tanpa nilai kosong.
> - **Berikutnya:** Eksplorasi pola.

In [ ]:
tx = pd.read_csv(DATA_DIR / "atm_withdrawals.csv", parse_dates=["date"])
master = pd.read_csv(DATA_DIR / "atm_master.csv")
ATM_TYPES = dict(zip(master["atm_id"], master["location_type"]))
CAPACITY = dict(zip(master["atm_id"], master["cassette_capacity"]))

assert tx.duplicated(["date", "atm_id"]).sum() == 0
assert (tx["withdrawal"] > 0).all()
completeness = tx.groupby("atm_id")["date"].agg(["min", "max", "count"])
assert (completeness["count"] == completeness["count"].max()).all(), "ada ATM dengan hari yang hilang"

W = to_wide(tx)
assert W.notna().all().all()
print(f"tx: {tx.shape} | master: {master.shape} | W (wide log1p): {W.shape} | {W.index.min().date()} – {W.index.max().date()}")
display(master)

### Kamus Data

| Kolom | File | Keterangan |
|---|---|---|
| `date` | withdrawals | tanggal (harian, lengkap tanpa lubang) |
| `atm_id` | keduanya | ID ATM (20 unit) |
| `withdrawal` | withdrawals | **target**: total penarikan tunai hari itu (juta Rp) |
| `location_type` | master | mall / office / residential / transport_hub / university / market |
| `city` | master | kota (informasi) |
| `cassette_capacity` | master | kapasitas maksimum kaset uang (juta Rp), dipakai di simulasi pengisian |

Target dimodelkan sebagai **log1p(penarikan)**, dengan tiga alasan: (1) pola musiman bersifat **multiplikatif** (gajian = ×1,4, bukan +Rp 40 jt), sehingga di skala log menjadi aditif; (2) ATM besar dan kecil bisa dilatih dalam satu model global; (3) varians lebih stabil.

## 3. Exploratory Data Analysis

> **🔄 Alur data: Deret Total & Tren**
>
> - **Input:** `W`.
> - **Proses:**
>   1. Total harian 20 ATM + rata-rata bergerak 28 hari, dengan garis Lebaran.
>   2. Indeks mingguan per tipe lokasi, dan total per tahun + perubahannya.
> - **Output:** Grafik. Total per tahun Rp 1.151 M → 1.100 M → 1.064 M → 1.011 M (−4,4%, −3,3%, −5,0%). Lonjakan besar setiap menjelang Lebaran.
> - **Berikutnya:** Bedah pola musiman.

In [ ]:
total = np.expm1(W).sum(axis=1)
fig, axes = plt.subplots(2, 1, figsize=(18, 8))
axes[0].plot(total.index, total, lw=0.6, color="#8DA0CB", label="total harian 20 ATM")
axes[0].plot(total.index, total.rolling(28, center=True).mean(), color="k", lw=2, label="rata-rata 28 hari")
for d in LEBARAN[LEBARAN <= total.index.max()]:
    axes[0].axvline(d, color="#C44E52", ls="--", lw=1)
axes[0].set_title("Total penarikan harian (juta Rp); garis merah = Lebaran")
axes[0].legend()
by_type = np.expm1(W).T.groupby(ATM_TYPES).sum().T.resample("W").mean()
(by_type / by_type.iloc[:8].mean()).plot(ax=axes[1], lw=1.2)
axes[1].set_title("Indeks penarikan mingguan per tipe lokasi (8 minggu pertama = 1)")
plt.tight_layout()
plt.show()

yearly = np.expm1(W).groupby(W.index.year).sum().sum(axis=1)
print("Total per tahun (juta Rp):", yearly.round(0).to_dict(), "| perubahan:", (yearly.pct_change() * 100).round(1).dropna().to_dict())

> **🔄 Alur data: Pola Mingguan, Gajian & Lebaran**
>
> - **Input:** `tx`, `master`, kalender.
> - **Proses:**
>   1. `long` = data panjang + kalender + `rel` (penarikan ÷ median bulanan ATM itu, sehingga ATM besar & kecil sebanding).
>   2. Heatmap median `rel` per tipe × hari, profil per tanggal (hari normal saja), dan *event study* −30…+21 hari terhadap Lebaran.
> - **Output:** 3 grafik. Kantor: akhir pekan ±0,4×; mall: Sabtu–Minggu ±1,35×. Lonjakan gajian tgl 25–2. Event study: naik bertahap 3 minggu sebelum Lebaran, lalu anjlok di H0.
> - **Berikutnya:** Dekomposisi satu ATM.

In [ ]:
long = tx.merge(master[["atm_id", "location_type"]], on="atm_id").merge(calendar_frame(W.index), on="date")
long["log_y"] = np.log(long["withdrawal"])
long["rel"] = long["withdrawal"] / long.groupby(["atm_id", long["date"].dt.to_period("M")])["withdrawal"].transform("median")

fig, axes = plt.subplots(1, 3, figsize=(22, 5))
dow_names = ["Sen", "Sel", "Rab", "Kam", "Jum", "Sab", "Min"]
dow_prof = long.pivot_table(index="location_type", columns="dow", values="rel", aggfunc="median")
dow_prof.columns = dow_names
sns.heatmap(dow_prof, annot=True, fmt=".2f", cmap="RdBu_r", center=1, ax=axes[0])
axes[0].set_title("Pola mingguan (relatif terhadap median bulanan ATM)")
dom_prof = long[(long["lebaran_bin"] == "normal") & (long["is_holiday"] == 0)].pivot_table(index="dom", columns="location_type", values="rel", aggfunc="median")
dom_prof.plot(ax=axes[1], lw=1.5)
for x in (2.5, 24.5):
    axes[1].axvline(x, color="k", ls=":")
axes[1].set_title("Pola tanggal (hari normal): lonjakan gajian tgl 25–2")
axes[1].set_xlabel("tanggal")
ev = long[long["days_to_lebaran"].between(-30, 21)].pivot_table(index="days_to_lebaran", columns="location_type", values="rel", aggfunc="median")
ev.plot(ax=axes[2], lw=1.5)
axes[2].axvline(0, color="#C44E52", ls="--")
axes[2].set_title("Event study Lebaran (4 tahun digabung): hari relatif terhadap Lebaran")
plt.tight_layout()
plt.show()

> **🔄 Alur data: Dekomposisi STL**
>
> - **Input:** `W[example_atm]` (ATM residential pertama).
> - **Proses:**
>   1. STL (periode 7, robust) → tren + musiman + residual.
>   2. Kekuatan musiman & tren = 1 − var(residual) / var(komponen + residual).
> - **Output:** Grafik STL. Kekuatan musiman mingguan **0,28**, tren **0,66**. Residual terbesar 0,85 (log) di sekitar Lebaran 2022 (29 Apr 2022).
> - **Berikutnya:** Uji statistik time series.

In [ ]:
example_atm = master.loc[master["location_type"] == "residential", "atm_id"].iloc[0]
stl = STL(W[example_atm], period=7, robust=True).fit()
strength_seasonal = max(0, 1 - stl.resid.var() / (stl.seasonal + stl.resid).var())
strength_trend = max(0, 1 - stl.resid.var() / (stl.trend + stl.resid).var())
fig = stl.plot()
fig.set_size_inches(16, 8)
fig.suptitle(f"STL decomposition log-penarikan {example_atm} (residential), periode 7 hari", y=1.02)
plt.tight_layout()
plt.show()
print(f"Kekuatan musiman mingguan = {strength_seasonal:.2f} | kekuatan tren = {strength_trend:.2f} (0 = tidak ada, 1 = sangat kuat)")
print(f"Residual STL masih memuat lonjakan besar di sekitar Lebaran: maks |resid| = {stl.resid.abs().max():.2f} pada {stl.resid.abs().idxmax().date()}")

### Insight EDA

- **Tren turun**: total penarikan 20 ATM turun dari Rp 1,15 T (2022) ke Rp 1,01 T (2025), sekitar −3% s.d. −5% per tahun, karena pembayaran beralih ke digital (QRIS, transfer). Model harus bisa mengikuti level yang terus turun.
- **Pola mingguan sangat berbeda per lokasi**: ATM kantor hanya ±0,4× di akhir pekan, mall naik ke ±1,35× di Sabtu–Minggu, kampus sepi di akhir pekan, sedangkan perumahan & pasar relatif datar. Satu pola mingguan untuk semua ATM tidak cukup.
- **Gajian**: penarikan melonjak pada tanggal 25 s.d. 2, paling kuat di ATM kantor & perumahan.
- **Lebaran adalah peristiwa terbesar dalam setahun**: penarikan naik bertahap 3 minggu sebelumnya (THR + mudik), memuncak di H-7…H-1 (pasar sampai ±2×), lalu **anjlok** saat Lebaran (kantor & kampus ±0,3×), dan pulih perlahan dalam 1–2 minggu. Tanggal Lebaran maju ±11 hari setiap tahun, sehingga pola musiman tahunan biasa tidak bisa menangkapnya.
- **STL** pada satu ATM perumahan: kekuatan musiman mingguan 0,28 dan tren 0,66. Residual terbesar (0,85 dalam log, ≈ ×2,3) terjadi di sekitar Lebaran. Artinya, komponen tren + musiman mingguan saja **tidak** menjelaskan lonjakan Lebaran, dan inilah titik lemah model klasik seperti Holt-Winters.

## 4. Uji Statistik Time Series

| Uji | Pertanyaan | H0 |
|---|---|---|
| **ADF** (Augmented Dickey-Fuller) | Apakah deret stasioner? | ada *unit root* (tidak stasioner) |
| **KPSS** | idem, dari arah sebaliknya | deret stasioner (di sekitar tren) |
| **Tren** (regresi log ~ waktu, SE Newey-West) | Apakah ada tren naik/turun nyata? | slope = 0 |
| **Kruskal-Wallis + ε²** | Apakah hari dalam seminggu berpengaruh (per tipe lokasi)? | median sama untuk semua hari |
| **Mann-Whitney + r** | Apakah gajian & pra-Lebaran berbeda dari hari biasa? | distribusi sama |
| **ACF/PACF** | Lag mana yang informatif? | — |

ADF dan KPSS dipakai berpasangan. Jika ADF menolak H0 **dan** KPSS tidak menolak H0, deret stasioner. Jika hasilnya bertentangan, deret “stasioner di sekitar tren/musim” dan perlu dimodelkan tren/musimannya.

> **🔄 Alur data: Stasioneritas & Tren per ATM**
>
> - **Input:** `W` (20 deret log).
> - **Proses:**
>   1. ADF (H0: unit root) dan KPSS dengan tren (H0: stasioner).
>   2. Regresi log ~ waktu dengan SE Newey-West (lag 14) → tren tahunan + p-value (Holm).
> - **Output:** `ts_tests`. Semua 20 deret stasioner di sekitar tren (ADF p < 0,001; KPSS p ≥ 0,10). **12 ATM** tren turun signifikan (Holm), median **−3,4%/tahun**.
> - **Berikutnya:** Uji efek kalender.

In [ ]:
rows = []
for atm in W.columns:
    s = W[atm]
    adf_p = adfuller(s, autolag="AIC", result_object=False)[1]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", InterpolationWarning)  # p-value KPSS dibatasi tabel 0,01–0,10
        warnings.simplefilter("ignore", FutureWarning)         # perubahan format output di statsmodels mendatang
        kpss_p = kpss(s, regression="ct", nlags="auto")[1]
    t = np.arange(len(s)) / 365.25
    ols = sm.OLS(s.to_numpy(), sm.add_constant(t)).fit(cov_type="HAC", cov_kwds={"maxlags": 14})
    rows.append({"atm_id": atm, "location_type": ATM_TYPES[atm], "adf_p": adf_p, "kpss_p": kpss_p,
                 "tren_per_tahun": np.exp(ols.params[1]) - 1, "tren_p": ols.pvalues[1]})
ts_tests = pd.DataFrame(rows).set_index("atm_id")
ts_tests["tren_p_holm"] = holm_correction(ts_tests["tren_p"])
ts_tests["kesimpulan_stasioneritas"] = np.select(
    [(ts_tests["adf_p"] < 0.05) & (ts_tests["kpss_p"] >= 0.05), (ts_tests["adf_p"] < 0.05) & (ts_tests["kpss_p"] < 0.05)],
    ["stasioner (di sekitar tren)", "stasioner + perubahan level/musim"], default="tidak stasioner")
with pd.option_context("display.float_format", "{:.4f}".format):
    display(ts_tests)
print("Ringkasan:", ts_tests["kesimpulan_stasioneritas"].value_counts().to_dict(),
      f"| tren signifikan turun (Holm): {((ts_tests['tren_p_holm'] < 0.05) & (ts_tests['tren_per_tahun'] < 0)).sum()} dari {len(ts_tests)} ATM,",
      f"median tren {ts_tests['tren_per_tahun'].median():+.2%}/tahun")

> **🔄 Alur data: Uji Efek Kalender & ACF**
>
> - **Input:** `long`.
> - **Proses:**
>   1. Per tipe lokasi, hari normal: residual = log penarikan − rata-rata ATM itu; Kruskal-Wallis + ε² untuk 7 hari dalam seminggu.
>   2. Mann-Whitney: jendela gajian vs tengah bulan, dan H-7…H-1 Lebaran vs hari normal (Holm).
>   3. ACF & PACF setelah differencing 7 hari.
> - **Output:** `cal_tests` + ACF/PACF. Efek hari: ε² kantor **0,63**, mall 0,58, kampus 0,54, transport 0,25, pasar 0,11, perumahan 0,07. Gajian vs tengah bulan: kantor +49% … kampus +2%. Pra-Lebaran vs normal: pasar **+85%**, perumahan +68%. Semua p_holm < 0,001.
> - **Berikutnya:** Rancang validasi.

In [ ]:
rows = []
normal = long[(long["lebaran_bin"] == "normal") & (long["is_holiday"] == 0)]
for loc, g in long.groupby("location_type"):
    n = normal[normal["location_type"] == loc]
    # residual = log penarikan − rata-rata ATM itu (di SEMUA hari), baru dikelompokkan per hari
    resid = n["log_y"] - n.groupby("atm_id")["log_y"].transform("mean")
    h, p = stats.kruskal(*[resid[n["dow"] == d] for d in range(7)])
    eps2 = h / ((len(n) ** 2 - 1) / (len(n) + 1))
    pay, mid = resid[n["dom_bin"].isin(["gajian_25-31", "gajian_1-2"])], resid[n["dom_bin"] == "tengah_10-19"]
    u = stats.mannwhitneyu(pay, mid)
    g_res = g["log_y"] - g.groupby("atm_id")["log_y"].transform("mean")
    pre, norm = g_res[g["lebaran_bin"] == "H-7_s.d._H-1"], g_res[(g["lebaran_bin"] == "normal") & (g["is_holiday"] == 0)]
    u2 = stats.mannwhitneyu(pre, norm)
    rows.append({"location_type": loc, "dow_eps2": eps2, "dow_p": p,
                 "gajian_vs_tengah_%": np.exp(pay.median() - mid.median()) - 1, "gajian_p": u.pvalue, "gajian_r": 2 * u.statistic / (len(pay) * len(mid)) - 1,
                 "pra_lebaran_vs_normal_%": np.exp(pre.median() - norm.median()) - 1, "lebaran_p": u2.pvalue})
cal_tests = pd.DataFrame(rows).set_index("location_type")
for col in ["dow_p", "gajian_p", "lebaran_p"]:
    cal_tests[col.replace("_p", "_p_holm")] = holm_correction(cal_tests[col])
with pd.option_context("display.float_format", "{:.4f}".format):
    display(cal_tests[["dow_eps2", "dow_p_holm", "gajian_vs_tengah_%", "gajian_r", "gajian_p_holm", "pra_lebaran_vs_normal_%", "lebaran_p_holm"]])

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
plot_acf(W[example_atm].diff(7).dropna(), lags=35, ax=axes[0], title=f"ACF {example_atm} (setelah differencing 7 hari)")
plot_pacf(W[example_atm].diff(7).dropna(), lags=35, ax=axes[1], method="ywm", title=f"PACF {example_atm} (setelah differencing 7 hari)")
plt.tight_layout()
plt.show()

### Interpretasi Uji Statistik

**Stasioneritas & tren**
- Semua 20 deret **stasioner di sekitar tren** (ADF menolak unit root, p < 0,001; KPSS tidak menolak stasioneritas-tren, p ≥ 0,10). Model tidak perlu differencing tambahan, cukup menangani level, tren, dan musim.
- **12 dari 20 ATM** punya tren turun yang signifikan (Holm, SE Newey-West), dengan median **−3,4% per tahun**. Delapan sisanya juga cenderung turun, tetapi tidak signifikan setelah koreksi.

**Efek kalender (semua signifikan, p_holm < 0,001)**
- **Hari dalam seminggu**: efek **besar** di kantor (ε² = 0,63), mall (0,58), kampus (0,54), dan sedang di transport hub (0,25). Di perumahan (0,07) dan pasar (0,11) efeknya kecil. Ini konsisten dengan heatmap di Bagian 3.
- **Gajian vs tengah bulan**: kantor **+49%**, perumahan +40%, pasar +22%, mall +21%, transport hub +14%, kampus hanya +2% (mahasiswa tidak menerima gaji tanggal 25).
- **Pra-Lebaran (H-7…H-1) vs hari normal**: pasar **+85%**, perumahan +68%, transport hub/mall/kantor/kampus +40–45%.
- **ACF/PACF** setelah differencing 7 hari menunjukkan korelasi signifikan di lag 1 dan lag kelipatan 7. Ini dasar pemilihan lag 1, 2, 3, 7, 14, 21, 28 di feature engineering.

**Kesimpulan untuk pemodelan:** ada tiga sumber variasi besar yang **berbeda per tipe lokasi** (hari, gajian, Lebaran) dan tidak periodik sederhana. Model perlu fitur kalender eksplisit, dan perlu bisa belajar interaksi “efek kalender × tipe lokasi”.

## 5. Desain Validasi: Rolling-Origin (Time-Series Cross-Validation)

Data deret waktu **tidak boleh di-split acak**. Split acak berarti model “melihat” hari esok saat dilatih untuk meramal hari ini. Validasi yang benar meniru pemakaian nyata:

```
origin 1:  [========== latih ==========] → ramal 14 hari
origin 2:  [============ latih ============] → ramal 14 hari
origin 3:  [============== latih ==============] → ramal 14 hari     (expanding window)
```

| Periode | Peran | Origin |
|---|---|---|
| 2022 | riwayat awal (tidak dievaluasi) | — |
| **2023** | **tuning** hyperparameter | 8 origin, tiap 42 hari |
| **2024** | **pemilihan model** + kalibrasi interval | 26 origin, tiap 14 hari (seluruh tahun tercakup tanpa tumpang tindih) |
| **2025** | **uji akhir** (backtest out-of-time), dikunci | 26 origin, tiap 14 hari |

Di setiap origin, **semua** model dilatih ulang dengan data ≤ origin, lalu meramal 14 hari. Total uji akhir: 26 × 14 hari × 20 ATM = **7.280 ramalan per model**.

> **🔄 Alur data: Origin Rolling-Origin**
>
> - **Input:** Rentang tanggal `W`.
> - **Proses:**
>   1. 8 origin tuning (2023), 26 origin seleksi (2024), 26 origin uji akhir (2025), tiap 14 hari.
>   2. `assert`: jendela 2024 tidak masuk 2025, dan jendela 2025 tidak melewati data.
> - **Output:** `TUNE_ORIGINS` (8: 15 Jan – 5 Nov 2023), `SELECT_ORIGINS` (26: 31 Des 2023 – 15 Des 2024), `TEST_ORIGINS` (26: 29 Des 2024 – 14 Des 2025). Lebaran 2025 masuk jendela origin 23 Mar 2025.
> - **Berikutnya:** Bangun fitur.

In [ ]:
TUNE_ORIGINS = pd.date_range("2023-01-15", periods=8, freq="42D")
SELECT_ORIGINS = pd.date_range("2023-12-31", periods=26, freq=f"{STEP}D")
TEST_ORIGINS = pd.date_range("2024-12-29", periods=26, freq=f"{STEP}D")
for name, o in [("tuning", TUNE_ORIGINS), ("seleksi", SELECT_ORIGINS), ("uji akhir", TEST_ORIGINS)]:
    last_day = o[-1] + pd.Timedelta(days=HORIZON)
    print(f"{name:<9}: {len(o)} origin, {o[0].date()} … {o[-1].date()} → ramalan s.d. {last_day.date()}")
assert TEST_ORIGINS[-1] + pd.Timedelta(days=HORIZON) <= W.index.max()
assert SELECT_ORIGINS[-1] + pd.Timedelta(days=HORIZON) < pd.Timestamp("2025-01-01")
lebaran_2025 = pd.Timestamp("2025-03-31")
covered = [o for o in TEST_ORIGINS if o < lebaran_2025 <= o + pd.Timedelta(days=HORIZON)]
print(f"Lebaran 2025 ({lebaran_2025.date()}) masuk jendela origin {covered[0].date()}; lonjakan pra-Lebaran tercakup origin sebelumnya")

## 6. Feature Engineering: Lag, Rolling, Binning Kalender + One-Hot

Fitur untuk tanggal **t** (semua dihitung dari log1p penarikan; lag/rolling hanya memakai data ≤ t−1):

| Kelompok | Fitur | Perlakuan |
|---|---|---|
| Lag | `lag_1`, `lag_2`, `lag_3`, `lag_7`, `lag_14`, `lag_21`, `lag_28` | angka |
| Rolling | rata-rata 7 & 28 hari, std 7 hari, rata-rata hari yang sama 4 minggu terakhir | angka |
| Hari & bulan | `dow` (Senin–Minggu), `month` | **one-hot** |
| Jendela gajian | tanggal → **bin**: gajian_1-2, awal_3-9, tengah_10-19, menjelang_20-24, gajian_25-31 | **binning → one-hot** |
| Fase Lebaran | jarak ke Lebaran → **bin**: H-21…H-15, H-14…H-8, H-7…H-1, H0…H+3, H+4…H+7, H+8…H+14, normal | **binning → one-hot** |
| Libur & musim | libur nasional, H-1/H+1 libur, puncak Desember, libur semester | 0/1 |
| ATM | tipe lokasi | **one-hot** |

**Kenapa binning?** Efek gajian tidak linear terhadap tanggal: lonjakan tgl 25 s.d. 2 lalu turun. Efek Lebaran juga berupa fase (THR, mudik, arus balik). Kalau dibiarkan sebagai angka, model butuh banyak percabangan untuk menemukan ambang yang sama. Binning dengan batas dari pengetahuan bisnis langsung memberi model “bahasa” yang tepat.

Fitur dibangun oleh **satu** fungsi (`src/features.py::build_features`) yang dipakai untuk training, forecast rekursif, dan API.

> **🔄 Alur data: Matriks Fitur**
>
> - **Input:** `W` s.d. 2024.
> - **Proses:**
>   1. `training_frame`: `build_features` untuk setiap tanggal yang punya 28 hari riwayat → lag, rolling, kalender, binning, one-hot; target = log1p hari itu.
>   2. Contoh satu baris: ATM residential pada H-5 Lebaran 2024.
> - **Output:** `X_demo` **21.360 × 53 fitur** (lag 7, rolling 4, flag 5, one-hot: bulan 12, hari 7, fase Lebaran 7, tipe lokasi 6, jendela gajian 5). Contoh ATM09 pada 5 Apr 2024: `lebaran_bin_H-7_s.d._H-1` = 1, `dow_4` (Jumat) = 1, `lag_1` = 5,30 (log), jauh di atas `lag_28` = 4,79 karena sudah masuk masa pra-Lebaran.
> - **Berikutnya:** Buktikan tidak ada kebocoran.

In [ ]:
X_demo, y_demo = training_frame(W.loc[:"2024-12-31"], ATM_TYPES)
print(f"Matriks training s.d. 2024: {X_demo.shape[0]:,} baris × {len(FEATURE_COLUMNS)} fitur")
groups = pd.Series(FEATURE_COLUMNS).str.extract(r"^(lag|roll|same|is|dow|month|dom_bin|lebaran_bin|location_type)")[0].value_counts()
print(groups.to_string())

row = X_demo[(X_demo["atm_id"] == example_atm) & (X_demo["date"] == "2024-04-05")].iloc[0]
print(f"\nContoh: {example_atm} pada 2024-04-05 (H-5 Lebaran, hari Jumat) — target log1p = {y_demo.loc[row.name]:.3f}")
display(row[FEATURE_COLUMNS][row[FEATURE_COLUMNS] != 0].to_frame("nilai").T.round(3))

> **🔄 Alur data: Uji Kebocoran**
>
> - **Input:** `W` dan salinannya yang dimodifikasi.
> - **Proses:**
>   1. Nilai aktual 12 Jun 2024 ditambah +5 (log), lalu fitur tanggal itu dibandingkan dengan versi asli (harus sama).
>   2. `lag_1` tanggal berikutnya harus sama dengan nilai yang dimodifikasi.
> - **Output:** ✅ Lulus: fitur hari t identik walaupun nilai aktual hari t diubah, dan `lag_1` hari t+1 = nilai hari t.
> - **Berikutnya:** Definisikan kandidat.

In [ ]:
# Uji kebocoran: mengubah nilai aktual pada hari t TIDAK boleh mengubah fitur hari t
W_leak = W.copy()
probe = pd.Timestamp("2024-06-12")
W_leak.loc[probe] = W_leak.loc[probe] + 5.0
f_orig = build_features(W, [probe], ATM_TYPES)[FEATURE_COLUMNS]
f_leak = build_features(W_leak, [probe], ATM_TYPES)[FEATURE_COLUMNS]
f_next = build_features(W_leak, [probe + pd.Timedelta(days=1)], ATM_TYPES)[["lag_1"]]
assert np.allclose(f_orig.to_numpy(), f_leak.to_numpy()), "BOCOR: fitur hari t berubah saat nilai hari t diubah"
assert np.allclose(f_next["lag_1"].to_numpy(), W_leak.loc[probe].to_numpy())
print("✅ Tidak ada kebocoran: fitur hari t hanya bergantung pada data ≤ t−1 (dan lag_1 hari t+1 = nilai hari t)")

## 7. Model

| | **Seasonal naive** (patokan) | **Holt-Winters / ETS** | **Gradient Boosting global** |
|---|---|---|---|
| Ide | ramalan = hari yang sama minggu lalu | level + tren teredam + musiman 7 hari, diperbarui eksponensial | pohon keputusan bertingkat pada fitur lag + kalender |
| Dilatih | — | **per ATM** (20 model), 2 tahun terakhir | **satu model untuk 20 ATM** (belajar pola bersama) |
| Kalender (gajian, Lebaran, libur) | ✗ | ✗ (hanya pola mingguan) | ✓ |
| Multi-step | langsung | langsung (rumus) | **rekursif**: ramalan hari ke-h menjadi lag untuk hari ke-h+1 |
| Kelebihan | sangat sederhana | klasik, interpretable, cepat | menangkap efek kalender & antar-ATM |

Pilihan *damped trend* ETS ditentukan per ATM dengan AIC. Seasonal naive juga menjadi penyebut **MASE**: MASE < 1 berarti lebih baik dari sekadar “sama dengan minggu lalu”.

> **🔄 Alur data: Grid Kandidat**
>
> - **Input:** Tidak ada data.
> - **Proses:**
>   1. 3 konfigurasi GBR (jumlah pohon, kedalaman, learning rate) dan 2 konfigurasi ETS (jendela latih 2 tahun / 1 tahun; damped dipilih per ATM via AIC).
>   2. Factory: fungsi yang membuat model baru (belum dilatih) untuk setiap origin.
> - **Output:** `GBR_GRID`, `ETS_GRID`, `TUNE_FACTORIES`.
> - **Berikutnya:** Tuning di 2023.

In [ ]:
GBR_GRID = {
    "gbr_a": dict(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.7),
    "gbr_b": dict(n_estimators=400, max_depth=4, learning_rate=0.05, subsample=0.7),
    "gbr_c": dict(n_estimators=600, max_depth=5, learning_rate=0.03, subsample=0.7),
}
ETS_GRID = {"ets_2y": dict(damped="auto", train_days=730), "ets_1y": dict(damped="auto", train_days=365)}


def gbr_factory(params):
    return lambda: GBRForecaster(ATM_TYPES, GradientBoostingRegressor(random_state=RANDOM_STATE, **params))


def ets_factory(params):
    return lambda: ETSForecaster(n_jobs=1, **params)


TUNE_FACTORIES = {**{k: gbr_factory(v) for k, v in GBR_GRID.items()}, **{k: ets_factory(v) for k, v in ETS_GRID.items()}}
print("Konfigurasi yang dituning:", list(TUNE_FACTORIES))

### 7.1 Model Statistik (Inferensi): Regresi Kalender per Tipe Lokasi + SARIMAX

Sebelum tuning dan backtesting (Bagian 8–11), efek kalender diukur secara **statistik** dengan intercept, koefisien, p-value, dan VIF. Dengan begitu kita tahu **berapa persen** gajian, Lebaran, atau akhir pekan mengubah penarikan, dan apakah efeknya **signifikan**.

**Model 1: regresi kalender per tipe lokasi** (data s.d. 31 Des 2024, tanpa menyentuh tahun uji 2025):

log(penarikan) = intercept + efek tetap ATM + hari + jendela gajian + fase Lebaran + bulan + libur + tren + error

- Dipisah **per tipe lokasi**, karena efek kalender berlawanan arah antar lokasi (akhir pekan: kantor turun, mall naik).
- **Efek tetap ATM** (dummy per ATM) menyerap perbedaan level antar ATM.
- Koefisien dummy → **efek % = e^koef − 1** terhadap acuan: Senin, tanggal 10–19, hari normal (bukan Lebaran), Januari, bukan libur.
- Data deret waktu panel: error **berautokorelasi** dan **berkorelasi antar ATM** di hari yang sama. SE biasa akan terlalu kecil, jadi dipakai **SE Driscoll-Kraay** (`hac-groupsum`, lag 7), yang tahan terhadap keduanya.

**Model 2: regresi dengan error ARIMA (SARIMAX)** untuk satu ATM: efek kalender di bagian regresi, sedangkan autokorelasi dimodelkan oleh komponen AR pada error. Ini model time series klasik dengan koefisien dan p-value. Dua spesifikasi dibandingkan dengan AIC.

> **🔄 Alur data: Regresi Kalender per Tipe Lokasi**
>
> - **Input:** `tx` s.d. 31 Des 2024 + `master` + kalender.
> - **Proses:**
>   1. `panel` = data harian per ATM + fitur kalender + log penarikan + tren (tahun).
>   2. `calendar_design`: dummy ATM/hari/gajian/Lebaran/bulan + flag libur + tren → `reference_coding` (acuan terbanyak).
>   3. Per tipe lokasi: `resolve_perfect_collinearity`, GVIF, OLS dengan SE Driscoll-Kraay (lag 7), lalu OLS biasa untuk diagnostik.
>   4. Ringkasan: R², GVIF maks, autokorelasi residual (rata-rata per ATM), rasio SE Driscoll-Kraay / SE biasa.
> - **Output:** ⟪Q_FIT⟫
> - **Berikutnya:** Baca efek kalender.

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

CAL_FLAGS = ["is_holiday", "is_pre_holiday", "is_post_holiday", "is_december_peak", "is_semester_break"]
panel = (tx[tx["date"] <= "2024-12-31"].merge(master[["atm_id", "location_type"]], on="atm_id")
         .merge(calendar_frame(W.index), on="date").sort_values(["date", "atm_id"]))
panel["log_y"] = np.log(panel["withdrawal"])
panel["t_years"] = (panel["date"] - W.index.min()).dt.days / 365.25
CAL_PREFIXES = {"dow": "dow_", "dom_bin": "dom_bin_", "lebaran_bin": "lebaran_bin_", "month": "month_", **{c: c for c in CAL_FLAGS}, "t_years": "t_years"}


def calendar_design(g, with_atm=True):
    cats = (["atm_id"] if with_atm else []) + ["dow", "dom_bin", "lebaran_bin", "month"]
    X = pd.get_dummies(g[cats].astype(str), dtype=float)
    X[CAL_FLAGS + ["t_years"]] = g[CAL_FLAGS + ["t_years"]].astype(float).to_numpy()
    X.index = g.index
    prefixes = ({"atm_id": "atm_id_"} if with_atm else {}) | CAL_PREFIXES
    return reference_coding(X, prefixes)


cal_models, cal_gvif, fit_rows = {}, {}, []
for loc, g in panel.groupby("location_type"):
    D, G, R = calendar_design(g)
    D, G, dropped = resolve_perfect_collinearity(D, G)
    gv = gvif(D, G)
    res = sm.OLS(g["log_y"].to_numpy(), sm.add_constant(D)).fit(cov_type="hac-groupsum", cov_kwds={"time": pd.factorize(g["date"])[0], "maxlags": 7})
    plain = sm.OLS(g["log_y"].to_numpy(), sm.add_constant(D)).fit()
    resid = pd.Series(plain.resid, index=g.index)
    acf1 = g.assign(r=resid).groupby("atm_id")["r"].apply(lambda r: r.autocorr(1)).mean()
    acf7 = g.assign(r=resid).groupby("atm_id")["r"].apply(lambda r: r.autocorr(7)).mean()
    cal_models[loc], cal_gvif[loc] = res, gv
    fit_rows.append({"location_type": loc, "n": int(res.nobs), "ATM": g["atm_id"].nunique(), "parameter": len(res.params),
                     "R²": res.rsquared, "R² adj": res.rsquared_adj, "kolinear_sempurna_dibuang": ", ".join(dropped) or "-",
                     "GVIF_adj_maks": gv["GVIF_adj"].max(), "ACF residual lag-1": acf1, "ACF residual lag-7": acf7,
                     "rasio SE Driscoll-Kraay / SE biasa (median)": float(np.median(res.bse / plain.bse))})
cal_fit = pd.DataFrame(fit_rows).set_index("location_type")
cal_fit.round(3)

> **🔄 Alur data: Efek Kalender (%), Tabel Koefisien & GVIF**
>
> - **Input:** `cal_models` (6 model).
> - **Proses:**
>   1. 17 efek kunci → % = e^koef − 1 per tipe lokasi, p-value Holm per model, heatmap dengan bintang.
>   2. Tabel lengkap model ATM kantor: intercept, koefisien, SE Driscoll-Kraay, p, CI, efek %; dan GVIF model kantor.
> - **Output:** ⟪Q_EFF⟫
> - **Berikutnya:** Model time series klasik untuk satu ATM.

In [ ]:
KEY_TERMS = {"Selasa": "dow_1", "Rabu": "dow_2", "Kamis": "dow_3", "Jumat": "dow_4", "Sabtu": "dow_5", "Minggu": "dow_6",
             "gajian tgl 25–31": "dom_bin_gajian_25-31", "gajian tgl 1–2": "dom_bin_gajian_1-2", "awal bulan 3–9": "dom_bin_awal_3-9",
             "H-21…H-15": "lebaran_bin_H-21_s.d._H-15", "H-14…H-8": "lebaran_bin_H-14_s.d._H-8", "H-7…H-1": "lebaran_bin_H-7_s.d._H-1",
             "H0…H+3 (Lebaran)": "lebaran_bin_H0_s.d._H+3", "H+4…H+7": "lebaran_bin_H+4_s.d._H+7",
             "libur nasional": "is_holiday", "puncak Desember": "is_december_peak", "tren per tahun": "t_years"}
eff = pd.DataFrame({loc: {k: np.exp(m.params[v]) - 1 for k, v in KEY_TERMS.items() if v in m.params} for loc, m in cal_models.items()})
pv = pd.DataFrame({loc: {k: m.pvalues[v] for k, v in KEY_TERMS.items() if v in m.params} for loc, m in cal_models.items()})
pv_holm = pv.apply(lambda col: holm_correction(col.dropna()).reindex(col.index))
star = pv_holm.map(lambda p: "" if pd.isna(p) else ("***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""))
fig, ax = plt.subplots(figsize=(13, 8))
sns.heatmap(eff * 100, annot=(eff * 100).round(0).astype("Int64").astype(str) + star, fmt="", cmap="RdBu_r", center=0, vmin=-80, vmax=80,
            ax=ax, cbar_kws={"label": "efek (%) vs acuan"})
ax.set_title("Efek kalender (%) per tipe lokasi — OLS + SE Driscoll-Kraay; * p_holm<0,05 (per tipe lokasi)")
plt.tight_layout()
plt.show()
display((eff * 100).round(1))

office = cal_models["office"]
ci = office.conf_int()
office_table = pd.DataFrame({"koefisien": office.params, "SE_Driscoll_Kraay": office.bse, "p_value": office.pvalues,
                             "ci95_lower": ci[0], "ci95_upper": ci[1], "efek_%": (np.exp(office.params) - 1) * 100})
office_table.index = office_table.index.str.replace("const", "INTERCEPT")
print(f"Contoh tabel lengkap — ATM kantor. INTERCEPT = log penarikan ATM acuan pada Senin, tgl 10–19, Januari, hari biasa, awal 2022 "
      f"→ Rp {np.exp(office.params['const']):,.1f} jt/hari")
with pd.option_context("display.max_rows", 60, "display.float_format", "{:.4f}".format):
    display(office_table)
print("GVIF model kantor:")
display(cal_gvif["office"].round(3).head(8))

> **🔄 Alur data: Regresi + Error ARIMA (SARIMAX, 1 ATM)**
>
> - **Input:** Deret `example_atm` 2022–2024, desain kalender tanpa dummy ATM.
> - **Proses:**
>   1. Regresi dengan error ARIMA (SARIMAX, konstanta di bagian regresi): dua spesifikasi, AR(1) + AR musiman 7 dan AR(1) saja, dibandingkan dengan OLS tanpa AR.
>   2. Perbandingan konvergensi, AIC, BIC, log-likelihood, Ljung-Box; spesifikasi dengan AIC terkecil dipilih.
>   3. Koefisien konstanta, AR, efek kalender (%), p-value, CI; ACF residual sebelum vs sesudah AR.
> - **Output:** ⟪Q_SAR⟫
> - **Berikutnya:** Interpretasi, lalu tuning rolling-origin (Bagian 8).

In [ ]:
sar_atm = example_atm
g = panel[panel["atm_id"] == sar_atm].sort_values("date")
D_sar, G_sar, _ = calendar_design(g, with_atm=False)
D_sar, G_sar, sar_dropped = resolve_perfect_collinearity(D_sar, G_sar)
endog = pd.Series(g["log_y"].to_numpy(), index=pd.DatetimeIndex(g["date"]).to_period("D"))
exog = sm.add_constant(D_sar.set_index(endog.index))      # regresi dengan error ARIMA: konstanta di bagian regresi

specs = {"AR(1) + AR musiman(7)": ((1, 0, 0), (1, 0, 0, 7)), "AR(1)": ((1, 0, 0), (0, 0, 0, 0))}
sar_fits = {name: SARIMAX(endog, exog=exog, order=o, seasonal_order=so, trend="n").fit(disp=False, maxiter=1000) for name, (o, so) in specs.items()}
ols_same = sm.OLS(endog.to_numpy(), exog.to_numpy()).fit()
sar_compare = pd.DataFrame({name: {"konvergen": bool(r.mle_retvals.get("converged")), "AIC": r.aic, "BIC": r.bic, "log-likelihood": r.llf,
                                   "Ljung-Box lag 14 p": float(ljung_box(r.resid, lags=(14,))["lb_pvalue"].iloc[0])} for name, r in sar_fits.items()})
sar_compare["OLS tanpa AR"] = {"konvergen": True, "AIC": ols_same.aic, "BIC": ols_same.bic, "log-likelihood": ols_same.llf,
                               "Ljung-Box lag 14 p": float(ljung_box(ols_same.resid, lags=(14,))["lb_pvalue"].iloc[0])}
assert all(sar_compare.loc["konvergen", list(specs)]), "SARIMAX tidak konvergen"
best_sar_name = sar_compare.loc["AIC", list(specs)].astype(float).idxmin()
sar_res = sar_fits[best_sar_name]
print(f"Regresi + error ARIMA untuk {sar_atm} ({ATM_TYPES[sar_atm]}), 2022–2024. Spesifikasi terpilih (AIC terkecil): {best_sar_name}")
display(sar_compare)

ci = sar_res.conf_int()
sar_table = pd.DataFrame({"koefisien": sar_res.params, "SE": sar_res.bse, "p_value": sar_res.pvalues, "ci95_lower": ci[0], "ci95_upper": ci[1]})
ts_terms = [t for t in ["const", "ar.L1", "ar.S.L7", "sigma2"] if t in sar_table.index]
show = ts_terms + [KEY_TERMS[k] for k in ["Sabtu", "Minggu", "gajian tgl 25–31", "H-7…H-1", "H0…H+3 (Lebaran)", "libur nasional", "tren per tahun"]
                   if KEY_TERMS[k] in sar_table.index]
sar_table["efek_%"] = np.where(sar_table.index.isin(ts_terms), np.nan, (np.exp(sar_table["koefisien"]) - 1) * 100)
print(f"ACF lag-1 residual: OLS tanpa AR {pd.Series(ols_same.resid).autocorr(1):.3f} → model terpilih {pd.Series(np.asarray(sar_res.resid)).autocorr(1):.3f}")
with pd.option_context("display.float_format", "{:.4f}".format):
    display(sar_table.loc[show])

#### Interpretasi Model Statistik

{FC_STAT_INTERPRETATION}

## 8. Tuning di 2023 (Rolling-Origin)

Setiap konfigurasi diuji di 8 origin tahun 2023. Kriteria: **WAPE terendah**. Konfigurasi terbaik untuk masing-masing keluarga model (GBR dan ETS) dibawa ke tahap pemilihan.

> **🔄 Alur data: Tuning 2023**
>
> - **Input:** `W`, 8 origin 2023, 5 konfigurasi.
> - **Proses:**
>   1. Untuk setiap origin (paralel): latih semua konfigurasi dengan data ≤ origin, lalu ramal 14 hari.
>   2. WAPE, bias, MAE, sMAPE per konfigurasi + std WAPE antar origin.
> - **Output:** `tuning`. GBR: gbr_c **12,1%** (600 pohon, depth 5, lr 0,03), gbr_b 12,4%, gbr_a 12,9%. ETS: 2 tahun 19,5%, 1 tahun 19,8%. Terpilih: `BEST_GBR = gbr_c`, `BEST_ETS = ets_2y`.
> - **Berikutnya:** Seleksi model di 2024.

In [ ]:
bt_tune = rolling_origin(TUNE_FACTORIES, W, TUNE_ORIGINS, HORIZON)
tuning = metrics_by(bt_tune, "model")[["wape", "bias", "mae", "smape"]]
tuning["keluarga"] = np.where(tuning.index.str.startswith("gbr"), "gbr", "ets")
per_origin = bt_tune.groupby(["model", "origin"]).apply(lambda g: forecast_metrics(g)["wape"], include_groups=False).unstack()
tuning["wape_std_antar_origin"] = per_origin.std(axis=1)
display(tuning.sort_values("wape").round(4))

BEST_GBR = tuning[tuning["keluarga"] == "gbr"]["wape"].idxmin()
BEST_ETS = tuning[tuning["keluarga"] == "ets"]["wape"].idxmin()
print(f"Terbaik: GBR = {BEST_GBR} {GBR_GRID[BEST_GBR]} | ETS = {BEST_ETS} {ETS_GRID[BEST_ETS]}")

## 9. Pemilihan Model di 2024 (Rolling-Origin, 26 Origin)

Tiga metode diuji di **seluruh tahun 2024** (26 jendela × 14 hari, tanpa tumpang tindih), termasuk Lebaran 2024.

**Aturan:** model dengan **WAPE 2024 terendah** dipilih. Signifikansi diuji dengan:
- **Diebold-Mariano** pada deret harian total absolute error (koreksi Newey-West untuk autokorelasi multi-step + koreksi sampel kecil HLN),
- **Wilcoxon signed-rank** pada WAPE per lintasan (origin × ATM, 520 pasangan).

> **🔄 Alur data: Backtest Seleksi 2024**
>
> - **Input:** `W`, 26 origin 2024, 3 metode (konfigurasi terbaik).
> - **Proses:**
>   1. Rolling-origin → `bt_2024` (ramalan, aktual, error, residual log).
>   2. Skala MASE per origin × ATM (MAE seasonal naive 365 hari sebelum origin).
>   3. Metrik per metode, lalu model dengan WAPE terendah dipilih (di luar naive).
> - **Output:** `bt_2024` (3 metode × 7.280 ramalan). WAPE: **GBR 12,3%**, ETS 21,9%, naive 24,9%; MASE GBR 0,57. `best_name = "gbr"`.
> - **Berikutnya:** Uji signifikansi & grafik.

In [ ]:
SELECT_FACTORIES = {"seasonal_naive": SeasonalNaive, "ets": ets_factory(ETS_GRID[BEST_ETS]), "gbr": gbr_factory(GBR_GRID[BEST_GBR])}
bt_2024 = rolling_origin(SELECT_FACTORIES, W, SELECT_ORIGINS, HORIZON)
bt_2024 = bt_2024.merge(naive_scale(W, SELECT_ORIGINS), on=["origin", "atm_id"])
sel = metrics_by(bt_2024, "model")
sel.index = sel.index.map(MODEL_LABELS)
display(sel.sort_values("wape").round(4))

best_name = metrics_by(bt_2024[bt_2024["model"] != "seasonal_naive"], "model")["wape"].idxmin()
other_name = "ets" if best_name == "gbr" else "gbr"
print(f"Model terpilih: {MODEL_LABELS[best_name]}")

> **🔄 Alur data: Uji Signifikansi 2024**
>
> - **Input:** `bt_2024`.
> - **Proses:**
>   1. `daily_loss`: total |error| per tanggal; `path_wape`: WAPE per lintasan origin × ATM.
>   2. Diebold-Mariano (Newey-West lag 14 + HLN) dan Wilcoxon untuk 3 pasangan.
>   3. Grafik WAPE per horizon & per origin.
> - **Output:** `sel_tests` + 2 grafik. GBR vs ETS: DM p ≈ 1,5×10⁻⁷, Wilcoxon p ≈ 10⁻⁵⁸, GBR lebih baik di 81% lintasan. Lonjakan error ETS terlihat jelas di sekitar Lebaran 2024.
> - **Berikutnya:** Kalibrasi interval.

In [ ]:
def daily_loss(bt, model):
    return bt[bt["model"] == model].groupby("date")["error"].apply(lambda e: e.abs().sum())


def path_wape(bt, model):
    g = bt[bt["model"] == model].groupby(["origin", "atm_id"])
    return g["error"].apply(lambda e: e.abs().sum()) / g["y"].sum()


def compare_models(bt, a, b):
    dm = diebold_mariano(daily_loss(bt, a), daily_loss(bt, b), h=HORIZON)
    pa, pb = path_wape(bt, a), path_wape(bt, b)
    w = stats.wilcoxon(pa, pb)
    return {"pembanding": f"{MODEL_LABELS[a]} vs {MODEL_LABELS[b]}", "DM_HLN": dm["DM_HLN"], "DM_p": dm["p_value"],
            "wilcoxon_p": w.pvalue, f"{MODEL_LABELS[a]}_lebih_baik_%_lintasan": (pa < pb).mean()}


sel_tests = pd.DataFrame([compare_models(bt_2024, best_name, other_name), compare_models(bt_2024, best_name, "seasonal_naive"),
                          compare_models(bt_2024, other_name, "seasonal_naive")]).set_index("pembanding")
with pd.option_context("display.float_format", "{:.4g}".format):
    display(sel_tests)

fig, axes = plt.subplots(1, 2, figsize=(17, 4.5))
for m, g in metrics_by(bt_2024, ["model", "h"])["wape"].groupby(level=0):
    axes[0].plot(g.index.get_level_values("h"), g.values * 100, "o-", label=MODEL_LABELS[m])
axes[0].set_xlabel("horizon (hari ke-)")
axes[0].set_ylabel("WAPE (%)")
axes[0].set_title("WAPE per horizon — 2024")
axes[0].legend()
for m in ["seasonal_naive", "ets", "gbr"]:
    w = bt_2024[bt_2024["model"] == m].groupby("origin").apply(lambda g: forecast_metrics(g)["wape"], include_groups=False)
    axes[1].plot(w.index, w.values * 100, "o-", label=MODEL_LABELS[m])
axes[1].axvspan(pd.Timestamp("2024-03-20"), pd.Timestamp("2024-04-24"), color="#C44E52", alpha=0.1, label="sekitar Lebaran")
axes[1].set_title("WAPE per origin — 2024")
axes[1].legend()
plt.tight_layout()
plt.show()

## 10. Interval Prediksi (Kalibrasi di 2024)

Ketidakpastian ramalan **membesar seiring horizon**. Interval dibuat dari **kuantil empiris residual log** (aktual − ramalan) model terpilih di backtest 2024, **per horizon**:

ramalan_q = ramalan × exp(kuantil residual_q, h)

Kuantil yang disimpan: P05, P10, P50, P90, P95. Interval 80% = [P10, P90] dan interval 90% = [P05, P95]. Kalibrasi ini **hanya** memakai 2024, lalu cakupannya diuji di 2025 (Bagian 11).

> **🔄 Alur data: Kuantil Interval per Horizon**
>
> - **Input:** Residual log model terpilih di 2024.
> - **Proses:**
>   1. Kuantil P05, P10, P50, P90, P95 per horizon 1–14 → `INTERVALS`.
> - **Output:** `INTERVALS` (14 horizon × 5 kuantil). Interval 80% ≈ ramalan × [0,80–0,84; 1,17–1,22]; interval 90% ≈ × [0,75–0,79; 1,24–1,31]. Lebarnya hampir konstan antar horizon, karena lag mingguan (7, 14, 21, 28) tetap tersedia dari data aktual.
> - **Berikutnya:** Uji akhir 2025.

In [ ]:
res_2024 = bt_2024[bt_2024["model"] == best_name]
INTERVALS = interval_quantiles(res_2024["log_resid"], res_2024["h"])
display(np.exp(INTERVALS).round(3).T)

fig, ax = plt.subplots(figsize=(10, 4))
for lo, hi, a in [("q05", "q95", 0.2), ("q10", "q90", 0.35)]:
    ax.fill_between(INTERVALS.index, (np.exp(INTERVALS[lo]) - 1) * 100, (np.exp(INTERVALS[hi]) - 1) * 100, alpha=a, color="#4C72B0",
                    label=f"{lo.upper()}–{hi.upper()}")
ax.plot(INTERVALS.index, (np.exp(INTERVALS["q50"]) - 1) * 100, "k-", label="median residual")
ax.axhline(0, color="grey", lw=0.8)
ax.set_xlabel("horizon (hari)")
ax.set_ylabel("aktual vs ramalan (%)")
ax.set_title(f"Lebar interval per horizon ({MODEL_LABELS[best_name]}, residual 2024)")
ax.legend()
plt.tight_layout()
plt.show()

## 11. Backtest Akhir 2025 (Out-of-Time)

Ketiga metode dijalankan ulang di 26 origin tahun 2025 dengan konfigurasi yang dipilih di 2023–2024. Tahun ini belum pernah dipakai untuk keputusan apa pun.

> **🔄 Alur data: Backtest Akhir 2025**
>
> - **Input:** `W`, 26 origin 2025 (dipakai pertama kali).
> - **Proses:**
>   1. Rolling-origin 3 metode → `bt_2025`; metrik per metode.
>   2. Cluster bootstrap 1.000× (resample lintasan origin × ATM) → 95% CI.
>   3. Forecast Value Added = pengurangan WAPE dibanding naive & ETS.
> - **Output:** `bt_2025`, `final_ci`. GBR: WAPE **11,57%** [11,27–11,85%], bias −0,15% [−0,78…+0,56%], MASE 0,53. ETS 19,9%, naive 22,1%. FVA: **−48%** vs naive, **−42%** vs ETS.
> - **Berikutnya:** Uji signifikansi 2025.

In [ ]:
bt_2025 = rolling_origin(SELECT_FACTORIES, W, TEST_ORIGINS, HORIZON)
bt_2025 = bt_2025.merge(naive_scale(W, TEST_ORIGINS), on=["origin", "atm_id"])
test_metrics = metrics_by(bt_2025, "model")
test_metrics.index = test_metrics.index.map(MODEL_LABELS)
display(test_metrics.sort_values("wape").round(4))

best_bt = bt_2025[bt_2025["model"] == best_name].copy()
final_ci = cluster_bootstrap_ci(best_bt, n_boot=1000, seed=RANDOM_STATE)
print(f"Cluster bootstrap 95% CI ({MODEL_LABELS[best_name]}, 1.000×, resample {best_bt.groupby(['origin', 'atm_id']).ngroups} lintasan):")
display(final_ci.round(4))

fva = pd.Series({f"vs {MODEL_LABELS[m]}": 1 - test_metrics.loc[MODEL_LABELS[best_name], "wape"] / test_metrics.loc[MODEL_LABELS[m], "wape"]
                 for m in ["seasonal_naive", other_name]})
print("Forecast Value Added (pengurangan WAPE):", (fva * 100).round(1).astype(str).add("%").to_dict())

> **🔄 Alur data: Uji Signifikansi & Stabilitas 2025**
>
> - **Input:** `bt_2025`.
> - **Proses:**
>   1. Diebold-Mariano & Wilcoxon untuk 3 pasangan model.
>   2. WAPE per origin dan uji tren Spearman (apakah performa memburuk sepanjang tahun?).
> - **Output:** `test_tests`. GBR vs ETS: DM = −8,65, p ≈ 1,6×10⁻¹⁶ (GBR lebih baik di 84% lintasan). GBR vs naive: p ≈ 1,9×10⁻³¹. WAPE per origin 9,9–14,7% (terburuk: jendela Lebaran), tren p = 0,12.
> - **Berikutnya:** Uji cakupan interval.

In [ ]:
test_tests = pd.DataFrame([compare_models(bt_2025, best_name, other_name), compare_models(bt_2025, best_name, "seasonal_naive"),
                           compare_models(bt_2025, other_name, "seasonal_naive")]).set_index("pembanding")
with pd.option_context("display.float_format", "{:.4g}".format):
    display(test_tests)

ow = best_bt.groupby("origin").apply(lambda g: forecast_metrics(g)["wape"], include_groups=False)
rho, p_trend = stats.spearmanr(np.arange(len(ow)), ow)
print(f"WAPE per origin 2025: rata-rata {ow.mean():.2%}, min {ow.min():.2%}, maks {ow.max():.2%} (origin {ow.idxmax().date()}); "
      f"tren Spearman ρ = {rho:+.2f}, p = {p_trend:.3f}")

> **🔄 Alur data: Cakupan Interval 2025**
>
> - **Input:** `best_bt`, `INTERVALS`.
> - **Proses:**
>   1. Untuk interval 80% (P10–P90) & 90% (P05–P95): % residual di dalam interval, uji binomial vs nominal, dan % di bawah/di atas.
>   2. Grafik WAPE per horizon, cakupan per horizon, dan contoh ramalan ±Lebaran 2025.
> - **Output:** `coverage`. P10–P90: **82,0%** (target 80%), P05–P95: **92,2%** (target 90%). Keduanya sedikit lebih lebar dari target (p < 0,001), dan meleset seimbang di bawah/atas (8,6%/9,3%).
> - **Berikutnya:** Cek bias per segmen.

In [ ]:
cov_rows = []
for lo, hi, nominal in [("q10", "q90", 0.80), ("q05", "q95", 0.90)]:
    q = INTERVALS.loc[best_bt["h"]]
    inside = (best_bt["log_resid"].to_numpy() >= q[lo].to_numpy()) & (best_bt["log_resid"].to_numpy() <= q[hi].to_numpy())
    best_bt[f"in_{lo}_{hi}"] = inside
    cov_rows.append({"interval": f"{lo.upper()}–{hi.upper()}", "nominal": nominal, **coverage_test(inside, nominal),
                     "di_bawah": (best_bt["log_resid"].to_numpy() < q[lo].to_numpy()).mean(),
                     "di_atas": (best_bt["log_resid"].to_numpy() > q[hi].to_numpy()).mean()})
coverage = pd.DataFrame(cov_rows).set_index("interval")
display(coverage.round(4))
cov_h = best_bt.groupby("h")[["in_q10_q90", "in_q05_q95"]].mean()

fig, axes = plt.subplots(1, 3, figsize=(22, 4.8))
for m, g in metrics_by(bt_2025, ["model", "h"])["wape"].groupby(level=0):
    axes[0].plot(g.index.get_level_values("h"), g.values * 100, "o-", label=MODEL_LABELS[m])
axes[0].set_title("WAPE per horizon — 2025")
axes[0].set_xlabel("horizon")
axes[0].legend()
axes[1].plot(cov_h.index, cov_h["in_q10_q90"] * 100, "o-", label="P10–P90 (target 80%)")
axes[1].plot(cov_h.index, cov_h["in_q05_q95"] * 100, "s-", label="P05–P95 (target 90%)")
for y in (80, 90):
    axes[1].axhline(y, color="k", ls=":")
axes[1].set_title("Cakupan interval per horizon — 2025")
axes[1].legend()
one = best_bt[best_bt["atm_id"] == example_atm].sort_values("date")
one = one[(one["date"] >= "2025-02-15") & (one["date"] <= "2025-05-15")]
axes[2].fill_between(one["date"], np.expm1(one["yhat_log"] + INTERVALS.loc[one["h"], "q10"].to_numpy()),
                     np.expm1(one["yhat_log"] + INTERVALS.loc[one["h"], "q90"].to_numpy()), alpha=0.3, label="P10–P90")
axes[2].plot(one["date"], one["yhat"], label="ramalan")
axes[2].plot(one["date"], one["y"], "k.", label="aktual")
axes[2].axvline(pd.Timestamp("2025-03-31"), color="#C44E52", ls="--", label="Lebaran")
axes[2].set_title(f"{example_atm}: ramalan backtest di sekitar Lebaran 2025")
axes[2].legend(fontsize=8)
plt.tight_layout()
plt.show()

> **🔄 Alur data: Bias per Segmen & Fase Lebaran**
>
> - **Input:** `best_bt` + kalender.
> - **Proses:**
>   1. `segment_bias` per tipe lokasi, fase Lebaran, jendela gajian, hari: WAPE, bias, t-test pada bias per lintasan (Holm).
>   2. Status: “bias material” jika signifikan **dan** |bias| > 2%; “signifikan tapi kecil” jika signifikan tetapi ≤ 2%.
>   3. WAPE per fase Lebaran: model terpilih vs ETS.
> - **Output:** `seg_all`. **0 segmen bias material**; 4 signifikan tapi kecil (0,3–0,7%). WAPE per fase Lebaran: GBR H0…H+3 **21,9%** vs ETS **52,3%**; H-7…H-1 12,2% vs 28,6%.
> - **Berikutnya:** Diagnostik residual 1 langkah.

In [ ]:
best_bt = best_bt.merge(calendar_frame(best_bt["date"].unique()), on="date").assign(location_type=lambda d: d["atm_id"].map(ATM_TYPES))
segments = {}
for col in ["location_type", "lebaran_bin", "dom_bin", "dow"]:
    segments[col] = segment_bias(best_bt, col)
seg_all = pd.concat(segments, names=["dimensi"])
seg_all["status"] = np.select([(seg_all["p_holm"] < 0.05) & (seg_all["bias"].abs() > 0.02), seg_all["p_holm"] < 0.05],
                              ["bias material (>2%)", "signifikan tapi kecil (≤2%)"], default="ok")
display(seg_all.round(4))

ets_bt = bt_2025[bt_2025["model"] == other_name].merge(calendar_frame(bt_2025["date"].unique()), on="date")
leb = pd.DataFrame({MODEL_LABELS[best_name]: metrics_by(best_bt, "lebaran_bin")["wape"], MODEL_LABELS[other_name]: metrics_by(ets_bt, "lebaran_bin")["wape"]})
print("WAPE per fase Lebaran 2025:")
display(leb.round(4))

### Interpretasi Backtest 2025

**Akurasi & signifikansi**
- Di 2025, Gradient Boosting mencapai **WAPE 11,6%** (95% CI cluster bootstrap 11,3–11,9%) dan **MASE 0,53**, yaitu error-nya hampir setengah dari seasonal naive. ETS: WAPE 19,9%; seasonal naive: 22,1%.
- **Forecast Value Added**: error berkurang **48%** dibanding naive dan **42%** dibanding Holt-Winters.
- Keunggulan ini **signifikan dan konsisten**: Diebold-Mariano vs ETS p ≈ 10⁻¹⁶, vs naive p ≈ 10⁻³¹. GBR lebih akurat di **84%** lintasan ATM × origin dibanding ETS, dan di 94% lintasan dibanding naive.
- Hasil 2025 (WAPE 11,6%) sejalan dengan hasil pemilihan di 2024 (12,3%), sehingga tidak ada tanda overfitting ke tahun pemilihan.
- **Tidak bias**: −0,15% (CI −0,8% s.d. +0,6%).
- **Stabil sepanjang tahun**: WAPE per origin 9,9–14,7%, tanpa tren memburuk (Spearman p = 0,12). Origin terburuk 23 Mar 2025 adalah jendela yang memuat hari Lebaran.

**Lebaran: di sinilah perbedaan terbesar**

| Fase | GBR | ETS |
|---|---|---|
| H-7…H-1 (puncak penarikan) | 12,2% | 28,6% |
| H0…H+3 (anjlok) | 21,9% | **52,3%** |
| H+4…H+7 | 16,6% | 43,5% |

ETS tidak tahu kapan Lebaran, jadi ramalannya meleset ke dua arah: kurang saat puncak (risiko ATM kosong) dan lebih saat anjlok (uang menganggur). GBR menangkapnya lewat fitur **binning fase Lebaran**. Fase H0…H+3 tetap yang tersulit (WAPE 22%), karena setiap tahun hanya ada 4 hari seperti ini.

**Interval prediksi**
- Cakupan P10–P90 **82,0%** (target 80%) dan P05–P95 **92,2%** (target 90%). Selisihnya 2 poin, signifikan secara statistik (n = 7.280), sehingga interval **sedikit terlalu lebar (konservatif)**. Penyebabnya, 2025 sedikit lebih mudah diramal dibanding 2024 (tahun kalibrasi). Untuk pengisian kas, interval yang sedikit konservatif lebih aman daripada yang terlalu sempit. Rekalibrasi tahunan akan mendekatkannya ke target.

**Bias per segmen**
- Tidak ada segmen dengan bias **material** (> 2%) yang signifikan. Empat segmen (Senin, Sabtu, tanggal 3–9, tanggal 10–19) **signifikan secara statistik tetapi kecil** (0,3–0,7%). Dengan ribuan observasi, selisih sekecil ini pun terdeteksi, padahal dampak bisnisnya bisa diabaikan.

## 12. Diagnostik Residual (Ramalan 1 Hari ke Depan)

Untuk menguji apakah masih ada pola yang belum tertangkap, model yang dilatih s.d. 31 Des 2024 meramal **setiap hari 2025 satu langkah ke depan** dengan lag aktual. Residualnya kemudian diuji:
- **Ljung-Box** (lag 7 & 14) per ATM, H0: residual tidak berautokorelasi. Jika ditolak, masih ada informasi waktu yang belum dipakai model.
- **ACF residual** gabungan.
- Distribusi residual (normalitas tidak wajib, karena interval memakai kuantil empiris).

> **🔄 Alur data: Ljung-Box Residual 1 Langkah**
>
> - **Input:** Model dilatih s.d. 2024, fitur 2025 dengan lag aktual.
> - **Proses:**
>   1. Residual log 1-langkah per ATM untuk 365 hari 2025.
>   2. Ljung-Box lag 7 & 14 per ATM (Holm), ACF lag-1, ACF plot, histogram.
> - **Output:** `lb_table`. Ljung-Box lag 14 signifikan (Holm) di 5 dari 20 ATM; median ACF lag-1 residual **0,10** (terbesar ATM01 0,26). Residual sedikit berekor tebal.
> - **Berikutnya:** Simulasi pengisian kas.

In [ ]:
diag_model = SELECT_FACTORIES[best_name]().fit(W, end="2024-12-31")
if best_name == "gbr":
    X25, y25 = training_frame(W, ATM_TYPES, start="2025-01-01")
    one_step = X25[["date", "atm_id"]].assign(resid=y25.to_numpy() - diag_model.predict_features(X25[FEATURE_COLUMNS].to_numpy(float)))
else:
    one_step = pd.concat([diag_model.fits_[a].forecast(365).pipe(lambda f: pd.DataFrame({"date": pd.date_range("2025-01-01", periods=365), "atm_id": a,
                         "resid": W.loc["2025", a].to_numpy()[:365] - f})) for a in W.columns])
lb_rows = []
for atm, g in one_step.groupby("atm_id"):
    lb = ljung_box(g.sort_values("date")["resid"], lags=(7, 14))
    lb_rows.append({"atm_id": atm, "location_type": ATM_TYPES[atm], "LB_7": lb["lb_stat"].iloc[0], "p_7": lb["lb_pvalue"].iloc[0],
                    "LB_14": lb["lb_stat"].iloc[1], "p_14": lb["lb_pvalue"].iloc[1], "acf_1": g.sort_values("date")["resid"].autocorr(1)})
lb_table = pd.DataFrame(lb_rows).set_index("atm_id")
lb_table["p_14_holm"] = holm_correction(lb_table["p_14"])
display(lb_table.round(4))
print(f"ATM dengan autokorelasi residual signifikan (lag 14, Holm): {(lb_table['p_14_holm'] < 0.05).sum()} dari {len(lb_table)} | "
      f"median ACF lag-1 residual = {lb_table['acf_1'].median():.3f}")

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
pooled = one_step.sort_values(["atm_id", "date"])
plot_acf(pooled[pooled["atm_id"] == example_atm]["resid"], lags=28, ax=axes[0], title=f"ACF residual 1-langkah {example_atm} (2025)")
axes[1].hist(one_step["resid"], bins=80, density=True, color="#4C72B0")
xs = np.linspace(one_step["resid"].min(), one_step["resid"].max(), 200)
axes[1].plot(xs, stats.norm.pdf(xs, one_step["resid"].mean(), one_step["resid"].std()), "r")
axes[1].set_title(f"Residual 1-langkah (log): skew {stats.skew(one_step['resid']):.2f}, kurtosis {stats.kurtosis(one_step['resid']):.2f}")
plt.tight_layout()
plt.show()

## 13. Simulasi Bisnis: Berapa Uang yang Diisi?

Setiap origin, bank merencanakan pengisian 14 hari ke depan. Untuk setiap kebijakan **isi = kuantil q dari distribusi ramalan** (dibatasi kapasitas kaset), dihitung di backtest 2025:
- **Cash-out**: hari ketika penarikan aktual > uang yang diisi.
- **Biaya menyimpan uang**: seluruh uang di kaset menanggung biaya dana + asuransi/keamanan (asumsi 8%/tahun).
- **Penalti cash-out**: asumsi Rp 2 jt per hari ATM kosong (fee transaksi hilang + reputasi).

Penyederhanaan: pengisian harian sesuai rencana. Di praktik pengisian dilakukan per 2–7 hari dan ada biaya *cash-in-transit* per kunjungan, tetapi logika trade-off-nya sama. **Biaya dan penalti adalah asumsi** dan harus diganti dengan angka internal bank. Karena itu disertakan **analisis sensitivitas** terhadap besarnya penalti.

Kuantil diambil dari residual **2024** (bukan 2025), lalu biayanya dihitung di 2025, sehingga pemilihan kuantil pun tidak bocor. Pembanding: kebijakan manual “**seasonal naive × 1,3**” (isi seperti minggu lalu + buffer 30%).

> **🔄 Alur data: Simulasi Pengisian Kas**
>
> - **Input:** `best_bt` (ramalan 2025), `INTERVALS`, kapasitas kaset, ramalan naive.
> - **Proses:**
>   1. 5 kebijakan: P50, P90, P95, P98, dan seasonal naive × 1,3 (manual). Isi dibatasi kapasitas.
>   2. Per kebijakan: % hari cash-out, rata-rata isi & sisa, biaya simpan (8%/th atas seluruh isi), penalti, total biaya.
>   3. Kurva biaya kuantil 0,500–0,995 (kuantil dari residual **2024**, biaya dihitung di 2025) → `LOAD_QUANTILE` optimal.
>   4. Sensitivitas: kuantil optimal untuk penalti Rp 0,5 / 1 / 2 / 5 jt dan penghematan dibanding kebijakan manual.
> - **Output:** `sim`, `cost_curve`, `sensitivity`. Manual (naive × 1,3): cash-out **19,1%** hari, biaya Rp 3.062 jt. P98: cash-out 1,7%, Rp 544 jt. Optimal kuantil **0,995**: cash-out 0,6%, Rp 425 jt (**−86%**). Hemat 64–92% di semua skenario penalti (Rp 0,5–5 jt).
> - **Berikutnya:** Interpretasi model.

In [ ]:
sim_base = best_bt[["origin", "atm_id", "date", "h", "y", "yhat_log"]].copy()
sim_base["capacity"] = sim_base["atm_id"].map(CAPACITY)
y_act, cap = sim_base["y"].to_numpy(), sim_base["capacity"].to_numpy()
naive_bt = bt_2025[bt_2025["model"] == "seasonal_naive"].set_index(["origin", "atm_id", "date"])["yhat"]
naive_load = naive_bt.loc[list(zip(sim_base["origin"], sim_base["atm_id"], sim_base["date"]))].to_numpy() * 1.3
resid_by_h = res_2024.groupby("h")["log_resid"]


def evaluate_policy(load_raw, penalty=CASHOUT_PENALTY):
    load = np.minimum(np.asarray(load_raw), cap)
    short = y_act > load
    carry = load.sum() * CARRY_COST / 365
    return {"cash_out_%": short.mean() * 100, "jumlah_cash_out": int(short.sum()), "rata2_isi_juta": load.mean(),
            "rata2_sisa_juta": np.clip(load - y_act, 0, None).mean(), "biaya_simpan_juta": carry,
            "penalti_juta": short.sum() * penalty, "total_biaya_juta": carry + short.sum() * penalty,
            "terkena_batas_kapasitas_%": (np.asarray(load_raw) > cap).mean() * 100}


def quantile_load(q):
    return np.expm1(sim_base["yhat_log"] + resid_by_h.quantile(q).loc[sim_base["h"]].to_numpy())


policies = {f"ramalan P{int(q * 100)}": quantile_load(q) for q in [0.50, 0.90, 0.95, 0.98]}
policies["seasonal naive × 1,3 (manual)"] = naive_load
sim = pd.DataFrame({k: evaluate_policy(v) for k, v in policies.items()}).T.sort_values("total_biaya_juta")
display(sim.round(2))

grid = np.round(np.arange(0.50, 0.9951, 0.005), 3)
loads = {q: quantile_load(q) for q in grid}
cost_curve = pd.DataFrame({q: evaluate_policy(l) for q, l in loads.items()}).T
LOAD_QUANTILE = float(cost_curve["total_biaya_juta"].idxmin())

sens = []
for pen in [0.5, 1.0, 2.0, 5.0]:
    c = pd.Series({q: evaluate_policy(l, pen)["total_biaya_juta"] for q, l in loads.items()})
    q_opt = c.idxmin()
    sens.append({"penalti_juta": pen, "kuantil_optimal": q_opt, "cash_out_%": cost_curve.loc[q_opt, "cash_out_%"],
                 "total_biaya_juta": c.min(), "biaya_manual_juta": evaluate_policy(naive_load, pen)["total_biaya_juta"]})
sensitivity = pd.DataFrame(sens).set_index("penalti_juta")
sensitivity["hemat_vs_manual_%"] = (1 - sensitivity["total_biaya_juta"] / sensitivity["biaya_manual_juta"]) * 100

fig, axes = plt.subplots(1, 2, figsize=(17, 4.5))
axes[0].plot(cost_curve.index, cost_curve["total_biaya_juta"], "-", label="total biaya")
axes[0].plot(cost_curve.index, cost_curve["biaya_simpan_juta"], "--", label="biaya simpan uang")
axes[0].plot(cost_curve.index, cost_curve["penalti_juta"], ":", label="penalti cash-out")
axes[0].axvline(LOAD_QUANTILE, color="#C44E52", ls="--", label=f"optimal q = {LOAD_QUANTILE:.3f}")
axes[0].set_xlabel("kuantil pengisian")
axes[0].set_ylabel("juta Rp (2025, 20 ATM)")
axes[0].set_title(f"Trade-off biaya (penalti Rp {CASHOUT_PENALTY:g} jt, biaya simpan {CARRY_COST:.0%}/th)")
axes[0].legend()
axes[1].plot(cost_curve["cash_out_%"], cost_curve["rata2_isi_juta"], "-")
for name, row in sim.iterrows():
    axes[1].scatter(row["cash_out_%"], row["rata2_isi_juta"], s=60, label=name)
axes[1].set_xlabel("hari cash-out (%)")
axes[1].set_ylabel("rata-rata uang diisi per ATM-hari (juta)")
axes[1].set_title("Frontier: tingkat layanan vs uang yang harus disiapkan")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()
print(f"Kuantil optimal = {LOAD_QUANTILE:.3f} → cash-out {cost_curve.loc[LOAD_QUANTILE, 'cash_out_%']:.2f}% hari, "
      f"total biaya {cost_curve.loc[LOAD_QUANTILE, 'total_biaya_juta']:,.0f} jt vs manual {sim.loc['seasonal naive × 1,3 (manual)', 'total_biaya_juta']:,.0f} jt")
print("Sensitivitas terhadap penalti cash-out:")
display(sensitivity.round(3))

## 14. Interpretasi Model

**Permutation importance per kelompok fitur**: semua kolom dalam satu kelompok (mis. seluruh one-hot fase Lebaran) diacak bersamaan, lalu dilihat kenaikan error ramalan 1 langkah di 2025. Cara ini lebih jujur daripada importance per kolom one-hot.

> **🔄 Alur data: Permutation Importance per Kelompok**
>
> - **Input:** Fitur 2025 (lag aktual), `diag_model`.
> - **Proses:**
>   1. 7 kelompok fitur diacak bersamaan (5 ulangan) → kenaikan MAE log.
> - **Output:** `group_imp`. Kenaikan MAE saat diacak: lag & rolling **+428%**, jendela gajian +27%, hari +21%, fase Lebaran +13%, libur & musim +11%, tipe lokasi +10%, bulan +0,4% (hampir tidak berguna setelah ada fitur lain).
> - **Berikutnya:** Rangkum semua bukti.

In [ ]:
if best_name == "gbr":
    X25m = X25[FEATURE_COLUMNS].to_numpy(float)
    base_err = np.abs(y25.to_numpy() - diag_model.predict_features(X25m)).mean()
    groups = {
        "lag & rolling": LAG_COLS,
        "hari (dow)": [c for c in FEATURE_COLUMNS if c.startswith("dow_")],
        "fase Lebaran (bin)": [c for c in FEATURE_COLUMNS if c.startswith("lebaran_bin_")],
        "jendela gajian (bin)": [c for c in FEATURE_COLUMNS if c.startswith("dom_bin_")],
        "bulan": [c for c in FEATURE_COLUMNS if c.startswith("month_")],
        "libur & musim": ["is_holiday", "is_pre_holiday", "is_post_holiday", "is_december_peak", "is_semester_break"],
        "tipe lokasi": [c for c in FEATURE_COLUMNS if c.startswith("location_type_")],
    }
    rng = np.random.default_rng(RANDOM_STATE)
    rows = []
    for gname, cols in groups.items():
        idx = [FEATURE_COLUMNS.index(c) for c in cols]
        incs = []
        for _ in range(5):
            Xp = X25m.copy()
            Xp[:, idx] = Xp[rng.permutation(len(Xp))][:, idx]
            incs.append(np.abs(y25.to_numpy() - diag_model.predict_features(Xp)).mean() / base_err - 1)
        rows.append({"kelompok": gname, "n_kolom": len(cols), "kenaikan_MAE_log_%": np.mean(incs) * 100, "std": np.std(incs) * 100})
    group_imp = pd.DataFrame(rows).set_index("kelompok").sort_values("kenaikan_MAE_log_%", ascending=False)
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.barh(group_imp.index, group_imp["kenaikan_MAE_log_%"], xerr=group_imp["std"], color="#4C72B0")
    ax.invert_yaxis()
    ax.set_xlabel("kenaikan MAE saat kelompok diacak (%)")
    ax.set_title("Permutation importance per kelompok fitur (ramalan 1 langkah 2025)")
    plt.tight_layout()
    plt.show()
    display(group_imp.round(2))
else:
    group_imp = pd.DataFrame()
    print("Model terpilih ETS: komponen = level, tren teredam, musiman 7 hari (lihat parameter di model.json)")

## 15. Ringkasan Evaluasi Statistik & Kesimpulan

> **🔄 Alur data: Ringkasan Evaluasi**
>
> - **Input:** Hasil Bagian 4–14.
> - **Proses:**
>   1. 15 pengujian vs patokan → ✅/⚠️/❌/ℹ️ (`evaluation_summary`), dan angka kunci → `EVALUATION_FOR_METADATA`.
> - **Output:** `evaluation_summary` (15 baris): **10 ✅, 3 ⚠️** (cakupan 2 interval, autokorelasi residual), 2 ℹ️, 0 ❌.
> - **Berikutnya:** Model produksi.

In [ ]:
def verdict(ok: bool, warn: bool = False) -> str:
    return "✅" if ok else ("⚠️" if warn else "❌")


bm = test_metrics.loc[MODEL_LABELS[best_name]]
t_other = test_tests.iloc[0]
t_naive = test_tests.iloc[1]
c80, c90 = coverage.loc["Q10–Q90"], coverage.loc["Q05–Q95"]
n_seg_bias = int((seg_all["status"] == "bias material (>2%)").sum())
n_seg_small = int((seg_all["status"] == "signifikan tapi kecil (≤2%)").sum())
n_lb = int((lb_table["p_14_holm"] < 0.05).sum())
rows = [
    ("Data", "ATM dengan tren turun signifikan", f"{((ts_tests['tren_p_holm'] < 0.05) & (ts_tests['tren_per_tahun'] < 0)).sum()} dari 20 (median {ts_tests['tren_per_tahun'].median():+.1%}/th)", "informasi", "ℹ️"),
    ("Data", "Efek gajian & pra-Lebaran signifikan", f"{(cal_tests['gajian_p_holm'] < 0.05).sum()}/6 & {(cal_tests['lebaran_p_holm'] < 0.05).sum()}/6 tipe lokasi", "informasi", "ℹ️"),
    ("Validasi", "Kebocoran fitur", "tidak ada (uji otomatis)", "lulus", verdict(True)),
    ("Pemilihan (2024)", f"{MODEL_LABELS[best_name]} vs {MODEL_LABELS[other_name]}", f"WAPE {sel.loc[MODEL_LABELS[best_name], 'wape']:.2%} vs {sel.loc[MODEL_LABELS[other_name], 'wape']:.2%}; DM p = {sel_tests.iloc[0]['DM_p']:.2g}", "p < 0,05", verdict(sel_tests.iloc[0]["DM_p"] < 0.05, True)),
    ("Uji akhir 2025", "WAPE", f"{final_ci.loc['wape', 'estimate']:.2%} (CI {final_ci.loc['wape', 'ci_lower']:.2%}–{final_ci.loc['wape', 'ci_upper']:.2%})", "≤ 15%", verdict(final_ci.loc["wape", "ci_upper"] <= 0.15, final_ci.loc["wape", "estimate"] <= 0.15)),
    ("Uji akhir 2025", "MASE", f"{bm['mase']:.3f}", "< 1 (lebih baik dari naive)", verdict(bm["mase"] < 1)),
    ("Uji akhir 2025", "Bias", f"{final_ci.loc['bias', 'estimate']:+.2%} (CI {final_ci.loc['bias', 'ci_lower']:+.2%}…{final_ci.loc['bias', 'ci_upper']:+.2%})", "|bias| < 3%", verdict(abs(final_ci.loc["bias", "estimate"]) < 0.03)),
    ("Uji akhir 2025", f"vs {MODEL_LABELS[other_name]} (Diebold-Mariano)", f"DM = {t_other['DM_HLN']:.2f}, p = {t_other['DM_p']:.2g}", "p < 0,05", verdict(t_other["DM_p"] < 0.05, True)),
    ("Uji akhir 2025", "vs seasonal naive (Diebold-Mariano)", f"DM = {t_naive['DM_HLN']:.2f}, p = {t_naive['DM_p']:.2g}; FVA {fva.iloc[0]:.0%}", "p < 0,05", verdict(t_naive["DM_p"] < 0.05)),
    ("Uji akhir 2025", "Stabilitas antar origin", f"WAPE {ow.min():.1%}–{ow.max():.1%}, tren p = {p_trend:.2f}", "tanpa tren memburuk", verdict(not (rho > 0 and p_trend < 0.05))),
    ("Interval", "Cakupan P10–P90 (target 80%)", f"{c80['coverage']:.1%} (p = {c80['p_value']:.3f})", "≈ 80% (± 3 poin)", verdict(c80["p_value"] > 0.05, abs(c80["coverage"] - 0.8) <= 0.03)),
    ("Interval", "Cakupan P05–P95 (target 90%)", f"{c90['coverage']:.1%} (p = {c90['p_value']:.3f})", "≈ 90% (± 3 poin)", verdict(c90["p_value"] > 0.05, abs(c90["coverage"] - 0.9) <= 0.03)),
    ("Residual", "Autokorelasi residual (Ljung-Box lag 14, Holm)", f"{n_lb} dari 20 ATM; median ACF lag-1 {lb_table['acf_1'].median():.2f}", "0 ATM, atau ACF kecil (< 0,2)",
     verdict(n_lb == 0, lb_table["acf_1"].median() < 0.2)),
    ("Residual", "Segmen dengan bias material (>2%, signifikan)", f"{n_seg_bias} dari {len(seg_all)} (+{n_seg_small} signifikan tapi ≤2%)", "0", verdict(n_seg_bias == 0, n_seg_bias <= 2)),
    ("Bisnis", f"Pengisian kuantil {LOAD_QUANTILE:.3f} vs manual (naive × 1,3)",
     f"biaya {cost_curve.loc[LOAD_QUANTILE, 'total_biaya_juta']:,.0f} vs {sim.loc['seasonal naive × 1,3 (manual)', 'total_biaya_juta']:,.0f} jt; cash-out {cost_curve.loc[LOAD_QUANTILE, 'cash_out_%']:.1f}% vs {sim.loc['seasonal naive × 1,3 (manual)', 'cash_out_%']:.1f}%",
     "lebih murah di semua skenario penalti", verdict((sensitivity["hemat_vs_manual_%"] > 0).all())),
]
evaluation_summary = pd.DataFrame(rows, columns=["aspek", "pengujian", "nilai", "patokan", "status"])
pd.set_option("display.max_colwidth", 130)
display(evaluation_summary.set_index(["aspek", "pengujian"]))
print("Rekap status:", evaluation_summary["status"].value_counts().to_dict())

EVALUATION_FOR_METADATA = {
    "selection_2024": {MODEL_LABELS[m]: float(v) for m, v in metrics_by(bt_2024, "model")["wape"].items()},
    "test_2025": {k: {"estimate": float(final_ci.loc[k, "estimate"]), "ci95": [float(final_ci.loc[k, "ci_lower"]), float(final_ci.loc[k, "ci_upper"])]}
                  for k in ["wape", "bias", "mase", "mae", "rmse"]},
    "test_2025_all_models_wape": {k: float(v) for k, v in test_metrics["wape"].items()},
    "diebold_mariano_p": {"vs_" + other_name: float(t_other["DM_p"]), "vs_seasonal_naive": float(t_naive["DM_p"])},
    "interval_coverage_2025": {"p10_p90": float(c80["coverage"]), "p05_p95": float(c90["coverage"])},
    "ljung_box_significant_atms": n_lb,
    "load_quantile": LOAD_QUANTILE,
    "cost_assumptions": {"carry_cost_per_year": CARRY_COST, "cashout_penalty_juta": CASHOUT_PENALTY},
    "load_quantile_sensitivity": {str(k): float(v) for k, v in sensitivity["kuantil_optimal"].items()},
}

### Kesimpulan Akhir

**Hasil: 10 ✅ · 3 ⚠️ · 0 ❌** (+2 baris informasi ℹ️)

**1. Gradient Boosting global layak dipakai untuk perencanaan kas ATM**

| Aspek | Bukti utama |
|---|---|
| **Akurasi** | WAPE 11,6% (CI 11,3–11,9%), MASE 0,53; error 48% lebih kecil dari naive dan 42% lebih kecil dari Holt-Winters |
| **Signifikansi** | Diebold-Mariano p < 10⁻¹⁵ (vs ETS) dan p < 10⁻³⁰ (vs naive), konsisten di tahun pemilihan (2024) dan uji akhir (2025) |
| **Tidak bias** | −0,15%, dan tidak ada segmen dengan bias material |
| **Stabil** | WAPE 9,9–14,7% di 26 origin, tanpa tren memburuk |
| **Saat kritis (Lebaran)** | WAPE 12–22% vs ETS 29–52% |
| **Nilai bisnis** | Pengisian kas berbasis ramalan menghemat **64–92%** biaya vs kebijakan manual, di **semua** skenario penalti |

**2. Tiga catatan ⚠️**
- **Interval sedikit konservatif** (cakupan 82% & 92%, target 80% & 90%). Ini aman untuk kas, tetapi perlu dikalibrasi ulang tahunan.
- **Autokorelasi residual** pada 5 dari 20 ATM (Ljung-Box). Besarnya kecil (median ACF lag-1 = 0,10), tetapi menunjukkan masih ada sedikit informasi waktu yang belum terpakai, misalnya kejutan (*shock*) beberapa hari terakhir. Pengembangan: tambah fitur residual/selisih lag, atau model *direct* per horizon.
- **Kuantil pengisian optimal mentok di 0,995** (batas atas yang masih bisa diestimasi andal dari 7.280 residual). Karena penalti ATM kosong jauh lebih mahal daripada biaya menyimpan uang, kebijakan optimal adalah **tingkat layanan sangat tinggi (±99–99,5%)**. Ini sejalan dengan target ketersediaan ATM di industri. Di praktik, batasnya ditentukan kapasitas kaset dan frekuensi kunjungan CIT, bukan rumus.

**3. Rekomendasi operasional**
1. Pakai **rekomendasi isi** dari API (kuantil optimal, dibatasi kapasitas). Untuk ATM yang sering mentok kapasitas, tambah frekuensi pengisian.
2. Kirim data aktual **harian** lewat `recent_actuals`, supaya lag selalu terbaru tanpa perlu retrain.
3. **Refit bulanan**, dan **rekalibrasi interval tahunan** dengan backtest rolling-origin seperti notebook ini.
4. **Perbarui kalender** (`src/calendar_id.py`) setiap tahun: tanggal Lebaran, Idul Adha, Imlek, dan cuti bersama. API memberi peringatan jika ramalan melewati batas kalender.
5. Ganti asumsi biaya (biaya simpan 8%/tahun, penalti Rp 2 jt) dengan angka internal bank, lalu jalankan ulang Bagian 13.

## 16. Model Produksi: Refit Semua Data, Export JSON & Ramalan Januari 2026

Konfigurasi terpilih dilatih ulang dengan **seluruh data s.d. 31 Des 2025**. Interval tetap memakai kuantil residual dari backtest 2024, yang sudah terbukti di 2025. Kuantil pengisian optimal dari Bagian 13 disimpan di metadata.

> **🔄 Alur data: Refit, Export JSON & Ramalan Jan 2026**
>
> - **Input:** Seluruh `W` (2022–2025).
> - **Proses:**
>   1. `final_model` dilatih di semua data; metadata (profil ATM, kuantil pengisian per horizon, evaluasi, batas validitas kalender).
>   2. `export_model`: pohon GBR (atau state ETS) + riwayat 35 hari terakhir + `INTERVALS` → `model.json`.
>   3. Ramalan 28 hari (1–28 Jan 2026) total 20 ATM.
> - **Output:** `final_model` (data s.d. 31 Des 2025), file **`models/model.json` (±1,4 MB)**, origin 31 Des 2025; grafik ramalan 1–28 Jan 2026.
> - **Berikutnya:** Validasi JSON.

In [ ]:
final_model = SELECT_FACTORIES[best_name]().fit(W)
now = datetime.now(timezone.utc)
metadata = {
    "model_name": best_name,
    "model_label": MODEL_LABELS[best_name],
    "model_version": now.strftime("%Y%m%d%H%M%S"),
    "trained_at": now.isoformat(timespec="seconds"),
    "training_period": f"{W.index.min().date()} – {W.index.max().date()}",
    "target": "log1p(penarikan harian), juta Rupiah",
    "validated_horizon_days": HORIZON,
    "max_horizon_days": 28,
    "calendar_valid_until": CALENDAR_VALID_UNTIL.strftime("%Y-%m-%d"),
    "params": GBR_GRID[BEST_GBR] if best_name == "gbr" else ETS_GRID[BEST_ETS],
    "atms": master.to_dict("records"),
    "load_quantile": LOAD_QUANTILE,
    "load_quantile_by_h": {int(h): float(v) for h, v in res_2024.groupby("h")["log_resid"].quantile(LOAD_QUANTILE).items()},
    "evaluation": EVALUATION_FOR_METADATA,
    "library_versions": {"python": platform.python_version(), "scikit-learn": sklearn.__version__, "pandas": pd.__version__, "numpy": np.__version__},
}
model_json = export_model(final_model, W, metadata, INTERVALS)
model_path = MODEL_DIR / "model.json"
save_json(model_json, model_path)
print(f"Model JSON -> {model_path} ({model_path.stat().st_size / 1024:.0f} KB), tipe {model_json['model']['type']}, origin {model_json['origin']}")

jan = final_model.forecast(W, W.index.max(), 28)
jan_total = np.expm1(jan.pivot(index="date", columns="atm_id", values="yhat_log")).sum(axis=1)
last = np.expm1(W.loc["2025-12-01":]).sum(axis=1)
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(last.index, last, "k-", label="aktual Des 2025")
ax.plot(jan_total.index, jan_total, "o-", color="#4C72B0", label="ramalan Jan 2026 (total 20 ATM)")
ax.set_title("Ramalan 28 hari ke depan dari model produksi")
ax.legend()
plt.tight_layout()
plt.show()

## 17. Validasi Model JSON & Contoh Request API

> **🔄 Alur data: Validasi JSON & Contoh Request**
>
> - **Input:** `model.json`, `final_model`, `W`.
> - **Proses:**
>   1. Ramalan 28 hari JSON vs model asli.
>   2. Simulasi data baru: 5 hari ramalan dianggap aktual → `recent_actuals` → ramalan 14 hari dari origin baru, dibandingkan dengan model asli.
>   3. Tulis `sample_request.json` & `sample_request_with_actuals.json`.
> - **Output:** Selisih maks JSON vs model asli **1,6×10⁻¹⁴**, baik untuk ramalan biasa maupun setelah update 5 hari data baru → identik. File contoh request ditulis.
> - **Berikutnya:** File contoh dipakai untuk API & unit test.

In [ ]:
json_model = JsonForecaster.load(model_path)
a = final_model.forecast(W, W.index.max(), 28).sort_values(["atm_id", "h"])
b = json_model.forecast(28).sort_values(["atm_id", "h"])
diff = np.abs(a["yhat_log"].to_numpy() - b["yhat_log"].to_numpy()).max()
print(f"Selisih maks ramalan 28 hari (JSON vs {MODEL_LABELS[best_name]} asli): {diff:.2e}")
assert diff < 1e-8
if best_name == "gbr":
    # simulasi data baru: anggap 5 hari pertama ramalan adalah aktual, lalu ramal lagi dari origin baru
    W_ext = pd.concat([W, np.log1p(np.expm1(a.pivot(index="date", columns="atm_id", values="yhat_log")).iloc[:5])])
    recent = {atm: {d.strftime("%Y-%m-%d"): float(np.expm1(W_ext.loc[d, atm])) for d in W_ext.index[-5:]} for atm in W.columns}
    c = json_model.forecast(14, recent_actuals=recent).sort_values(["atm_id", "h"])
    d_ = final_model.forecast(W_ext, W_ext.index.max(), 14).sort_values(["atm_id", "h"])
    diff2 = np.abs(c["yhat_log"].to_numpy() - d_["yhat_log"].to_numpy()).max()
    print(f"Selisih maks setelah update 5 hari data baru: {diff2:.2e}")
    assert diff2 < 1e-8
print("✅ Model JSON identik dengan model asli")

sample = {"atm_ids": [example_atm, "ATM01"], "horizon": 14}
(PROJECT_DIR / "sample_request.json").write_text(json.dumps(sample, indent=2))
if best_name == "gbr":
    recent_one = {atm: dict(list(v.items())[:3]) for atm, v in recent.items()}
    (PROJECT_DIR / "sample_request_with_actuals.json").write_text(json.dumps({"horizon": 7, "recent_actuals": recent_one}, indent=2))
b[b["atm_id"] == example_atm].head(7).set_index("date").drop(columns="atm_id").round(2)

## 18. Alur Data di API

```
POST /forecast  {atm_ids?, horizon (1–28, default 14), recent_actuals?}
   │ 1. Pydantic: ATM dikenal, horizon 1–28, recent_actuals berurutan mulai origin+1 untuk SEMUA ATM
   ▼
   │ 2. riwayat 35 hari di model.json (+ recent_actuals jika ada) → wide log1p
   │ 3. rekursif h = 1..horizon: build_features (lag, rolling, binning kalender → one-hot) → pohon GBR
   │    (atau rumus Holt-Winters jika model ETS) → log-ramalan
   │ 4. interval: ramalan × exp(kuantil residual per horizon)
   │ 5. rekomendasi isi = ramalan × exp(kuantil optimal per horizon), dibatasi kapasitas kaset
   ▼
Response per ATM: tanggal, ramalan, P10/P90, P05/P95, rekomendasi isi, penanda kalender,
                  total 14 hari, peringatan (horizon > 14, kalender, kapasitas)
```

**Siklus berikutnya:** setiap hari kirim data aktual terbaru lewat `recent_actuals` (atau refit model). Lakukan **refit bulanan** agar tren dan pola terbaru terserap, dan perbarui `src/calendar_id.py` setiap tahun (tanggal Lebaran, Idul Adha, Imlek, cuti bersama).